# `render_improve.ipynb` — overview

This notebook is the **multi-view rendering + dataset cleanup + foreground-mask** sidecar to `facelift_pipeline.ipynb`. It is self-contained: every section can be run with a fresh kernel, no upstream cell dependencies inside this file.

## What the notebook produces

1. **All-map renders** (RGB + depth-16bit + normal + opacity, 1024×1024) for every splat in `data/splats/`, written under `data/{rgb_rendered,depth_maps,normal_maps,opacity_maps}/`. The depth/normal renders go through the same FaceLift CUDA 3DGS rasterizer as RGB so everything is pixel-aligned.
2. **Multi-view consistency stats** for each splat at yaw = 0° vs yaw = +15°, with shared global depth normalization (single `(d_min, d_max)` across views — *not* per-view), tight visibility masks, and a per-sample `cons_rate / mean|Δz|` row in `data/multiview_preview/consistency_summary.csv`. Used to set the empirical Zone-A boundary for zone-aware Depth SR evaluation.
3. **Foreground masks** built from `data/cropped_faces/<stem>.png` (white-background avatars) → non-white pixels → morphological close+open → largest CC → NEAREST-resampled to **1024×1024 (`mask/`) and 256×256 (`mask_lr/`)**. Consumed by `scripts/train_depth_upres.py` as the **training loss mask** and the **val L1 metric mask**, so neither the loss nor the eval number is dominated by the white background.
4. **Mask debug visualizer** — 8-panel `RGB / opacity / depth / mask_op / mask_face / mask_final / RGB+boundary / depth*mask_final` collage per sample, sorted by `keep_r = |mask_final| / |mask_face|` so the worst samples sit at the top of `mask_debug_summary.csv`.

## Section index

| § | Type | What it does |
|---|------|--------------|
| 0 | setup | Load `configs/pipeline_config.yaml`, set `PROJECT_ROOT` |
| 4.1 | render | Build the GPU rasterizer wrapper (`render_all_maps_gpu` + deps) |
| 4.2 | render | Render every splat → RGB / depth / normal / opacity |
| 4.3 | utility | Re-scan `data/raw_faces` for postprocess refresh |
| 5.1 | camera | `get_camera_at_yaw(yaw_deg)` — generalized off-axis camera |
| 5.2 | render | Render one sample at multiple yaws with **shared global depth range** |
| 5.3 | metric | `reproject_consistency` — multi-view reprojection + tight FG mask |
| 5.4 | batch | Run 5.2+5.3 over all splats, write `consistency_summary.csv` |
| A | cleanup | Scan low-consistency samples, on-demand re-render preview |
| B | cleanup | Manual-confirm batch deletion of bad samples |
| 6.1 | mask | **Build foreground masks from `cropped_faces` → 1024 + 256** |
| 6.2 | debug | 8-panel mask debug per sample + `mask_debug_summary.csv` |

## Conventions / gotchas

- **Depth normalization**: `render_all_maps_gpu` defaults to *per-view* min/max, which makes raw depths from different cameras incomparable. For any multi-view experiment use the `_render_views_shared_range` helper in §5.2 — it computes one `(d_min, d_max)` across all requested cameras first.
- **Zone-aware eval**: report SR metrics only inside FaceLift's confidence cone (`|yaw| ≤ 20°`). The actual boundary is set empirically from §5.4's `cons_rate` distribution, not the placeholder ±20°.
- **Mask is binary**: both 1024 and 256 mask resamples use `INTER_NEAREST` on purpose so the loss has hard face/background edges, not feathered fractional weights.
- **Yaw sign**: positive `yaw_deg` in §5.1 offsets azimuth by `+yaw_deg` from frontal (azimuth=270). Subject-left vs subject-right hasn't been verified empirically — sanity-check on one sample before trusting batch results.
- **Self-contained**: §6.1 and §6.2 do not depend on `config` or any earlier rendering cell — only `cv2 / numpy / PIL` and the `data/` folders relative to the project root, so they can be run on a cold kernel.

## Downstream consumer

`scripts/train_depth_upres.py` (DepthUpResUNet) reads the masks via `DepthUpResDataset`:

- **Training loss**: `depth_loss(pred, hr_target, mask=mask)` — masked L1 + masked gradient loss, where a gradient pixel is only valid if both neighbours are inside the mask.
- **Val metric**: `val_l1 = _masked_l1(pred, hr_target, mask)` — same masked L1 used for training, so train/val numbers are directly comparable.
- **Fallback**: if `mask/` is missing the dataset prints a one-time warning and uses an all-ones mask, so the trainer still runs without §6.1 having been executed.


## 0. Setup — load config + project paths


In [ ]:
# Section 0 — Setup
# Run this cell ONCE per kernel session before anything else. It loads the
# project config, computes every shared path / parameter, and brings in the
# common imports used across the rest of the notebook so individual sections
# don't have to re-declare them.
from pathlib import Path
import os, sys, csv, json, time, shutil, subprocess
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# ---- Project root (notebook lives at FaceLift/, project root is one level up) ----
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "FaceLift" else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)   # so relative paths in the YAML resolve correctly

# ---- Config ----
CONFIG_PATH = PROJECT_ROOT / "configs" / "pipeline_config.yaml"
assert CONFIG_PATH.exists(), f"Missing {CONFIG_PATH}"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# ---- FaceLift repo on sys.path (for gslrm imports in section 1.1) ----
FACELIFT_REPO_PATH = Path(config["paths"]["facelift_repo"]).resolve()
if str(FACELIFT_REPO_PATH) not in sys.path:
    sys.path.insert(0, str(FACELIFT_REPO_PATH))

# ---- Common data paths (single source of truth — used everywhere downstream) ----
DATA_ROOT         = (PROJECT_ROOT / "data").resolve()
SPLAT_DIR         = Path(config["paths"]["splat_output"]).resolve()
DEPTH_DIR         = Path(config["paths"]["depth_output"]).resolve()
NORMAL_DIR        = Path(config["paths"]["normal_output"]).resolve()
OPACITY_DIR       = Path(config["paths"]["opacity_output"]).resolve()
RGB_DIR           = Path(config["paths"]["rgb_output"]).resolve()
CROPPED_FACES_DIR = (DATA_ROOT / "cropped_faces").resolve()
MULTIVIEW_DIR     = (DATA_ROOT / "multiview_preview").resolve()
DATASET_DIR       = (DATA_ROOT / "dataset").resolve()

# ---- Render parameters from config ----
RENDER_RES = config["depth_export"]["render_resolution"]
CAM_DIST   = config["depth_export"]["camera_distance"]
FOV        = config["depth_export"]["fov"]

print(f"PROJECT_ROOT      = {PROJECT_ROOT}")
print(f"CONFIG_PATH       = {CONFIG_PATH}")
print(f"DATA_ROOT         = {DATA_ROOT}")
print(f"SPLAT_DIR         = {SPLAT_DIR}            (exists={SPLAT_DIR.exists()})")
print(f"MULTIVIEW_DIR     = {MULTIVIEW_DIR}        (exists={MULTIVIEW_DIR.exists()})")
print(f"DATASET_DIR       = {DATASET_DIR}          (exists={DATASET_DIR.exists()})")
print(f"FACELIFT_REPO_PATH = {FACELIFT_REPO_PATH}  (added to sys.path)")
print(f"RENDER_RES={RENDER_RES}  CAM_DIST={CAM_DIST}  FOV={FOV}")


## 1.1 GPU rasterizer wrapper — RGB + Depth + Normal + Opacity


In [ ]:
import torch
import math, copy, sys

# Add FaceLift repo to path for gslrm imports
if str(FACELIFT_REPO_PATH) not in sys.path:
    sys.path.insert(0, str(FACELIFT_REPO_PATH))

from gslrm.model.gaussians_renderer import (
    GaussianModel as GS_GaussianModel,
    GaussianRasterizationSettings,
    GaussianRasterizer,
    Camera,
    render_opencv_cam,
)

RENDER_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def get_frontal_camera(hfov=50, w=512, h=512, radius=2.7, elevation=0):
    """
    Get frontal-view camera (FaceLift convention: azimuth=270 = front face).
    Returns: fxfycxcy_tensor, c2w_tensor
    """
    fx = w / (2 * np.tan(np.deg2rad(hfov) / 2.0))
    fy = fx
    cx, cy = w / 2.0, h / 2.0
    fxfycxcy = np.array([fx, fy, cx, cy], dtype=np.float32)

    elev = np.deg2rad(elevation)
    azim = np.deg2rad(270)  # FaceLift front view
    z = radius * np.sin(elev)
    base = radius * np.cos(elev)
    x = base * np.cos(azim)
    y = base * np.sin(azim)
    cam_pos = np.array([x, y, z])

    up_vector = np.array([0, 0, 1])
    forward = -cam_pos / np.linalg.norm(cam_pos)
    right = np.cross(forward, up_vector)
    right = right / np.linalg.norm(right)
    up = np.cross(right, forward)
    up = up / np.linalg.norm(up)

    R = np.stack((right, -up, forward), axis=1)
    c2w = np.eye(4, dtype=np.float32)
    c2w[:3, :4] = np.concatenate((R, cam_pos[:, None]), axis=1)

    fxfycxcy_t = torch.from_numpy(fxfycxcy).float().to(RENDER_DEVICE)
    c2w_t = torch.from_numpy(c2w).float().to(RENDER_DEVICE)
    return fxfycxcy_t, c2w_t


def render_with_custom_colors(pc, height, width, c2w, fxfycxcy, colors_precomp, bg_color=(0.0, 0.0, 0.0)):
    """
    Render using the CUDA rasterizer but with custom per-Gaussian colors.
    This lets us render depth/normal maps through the same pipeline as RGB.
    """
    viewpoint_camera = Camera(C2W=c2w, fxfycxcy=fxfycxcy, h=height, w=width)
    bg = torch.tensor(list(bg_color), dtype=torch.float32, device=c2w.device)

    raster_settings = GaussianRasterizationSettings(
        image_height=int(viewpoint_camera.h),
        image_width=int(viewpoint_camera.w),
        tanfovx=viewpoint_camera.tanfovX,
        tanfovy=viewpoint_camera.tanfovY,
        bg=bg,
        scale_modifier=1.0,
        viewmatrix=viewpoint_camera.world_view_transform,
        projmatrix=viewpoint_camera.full_proj_transform,
        sh_degree=0,  # No SH - using precomputed colors
        campos=viewpoint_camera.camera_center,
        prefiltered=False,
        debug=False,
    )

    rasterizer = GaussianRasterizer(raster_settings=raster_settings)

    rendered_image, radii = rasterizer(
        means3D=pc.get_xyz,
        means2D=torch.zeros_like(pc.get_xyz[:, :2], requires_grad=False),
        shs=None,
        colors_precomp=colors_precomp,
        opacities=pc.get_opacity,
        scales=pc.get_scaling,
        rotations=pc.get_rotation,
        cov3D_precomp=None,
    )

    return rendered_image  # (3, H, W)


def render_all_maps_gpu(ply_path, fxfycxcy, c2w, width=512, height=512):
    """
    Render RGB + Depth + Normal + Opacity using GPU for ALL maps.
    """
    pc = GS_GaussianModel(sh_degree=3)
    pc.load_ply(str(ply_path))
    pc = pc.to(RENDER_DEVICE)

    with torch.no_grad():
        # --- 1) RGB via standard SH rendering ---
        result_rgb = render_opencv_cam(pc, height, width, c2w, fxfycxcy, bg_color=(1.0, 1.0, 1.0))
        rgb = result_rgb["render"].detach().cpu().numpy()
        rgb = (rgb * 255).clip(0, 255).astype(np.uint8).transpose(1, 2, 0)

        # --- 2) Opacity: render with all-white color, black bg ---
        n_pts = pc.get_xyz.shape[0]
        white = torch.ones(n_pts, 3, dtype=torch.float32, device=RENDER_DEVICE)
        opacity_render = render_with_custom_colors(
            pc, height, width, c2w, fxfycxcy, white, bg_color=(0.0, 0.0, 0.0)
        )
        opacity_map = opacity_render[0].detach().cpu().numpy()  # Use R channel
        opacity_out = (opacity_map.clip(0, 1) * 255).astype(np.uint8)

        # --- 3) Depth: encode camera-space Z as color ---
        viewpoint_camera = Camera(C2W=c2w, fxfycxcy=fxfycxcy, h=height, w=width)
        w2c = viewpoint_camera.world_view_transform.T  # (4, 4)
        xyz_world = pc.get_xyz  # (N, 3)
        xyz_h = torch.cat([xyz_world, torch.ones(n_pts, 1, device=RENDER_DEVICE)], dim=1)
        xyz_cam = (w2c @ xyz_h.T).T[:, :3]  # (N, 3)
        depths = xyz_cam[:, 2]  # camera-space Z

        # Normalize depth to [0, 1] for rendering
        valid_mask = depths > 0.1
        if valid_mask.any():
            d_min = depths[valid_mask].min()
            d_max = depths[valid_mask].max()
            if d_max > d_min:
                depth_norm = 1.0 - (depths - d_min) / (d_max - d_min)  # invert: closer = brighter
            else:
                depth_norm = torch.zeros_like(depths)
        else:
            depth_norm = torch.zeros_like(depths)
        depth_norm = depth_norm.clamp(0, 1)

        depth_colors = depth_norm.unsqueeze(1).expand(-1, 3)  # (N, 3) grayscale
        depth_render = render_with_custom_colors(
            pc, height, width, c2w, fxfycxcy, depth_colors, bg_color=(0.0, 0.0, 0.0)
        )
        depth_map = depth_render[0].detach().cpu().numpy()  # R channel = normalized depth
        depth_map = depth_map.clip(0, 1)

        # --- 4) Normal: encode camera-space normals as color ---
        # Compute normals from Gaussian rotation (min-scale axis)
        rotations = pc.get_rotation  # (N, 4) quaternions
        scales = pc.get_scaling  # (N, 3) log scales

        # Convert quaternion to rotation matrix
        q = rotations / (rotations.norm(dim=1, keepdim=True) + 1e-8)
        w_q, x_q, y_q, z_q = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
        R_mat = torch.stack([
            1 - 2*(y_q*y_q + z_q*z_q), 2*(x_q*y_q - w_q*z_q),     2*(x_q*z_q + w_q*y_q),
            2*(x_q*y_q + w_q*z_q),     1 - 2*(x_q*x_q + z_q*z_q), 2*(y_q*z_q - w_q*x_q),
            2*(x_q*z_q - w_q*y_q),     2*(y_q*z_q + w_q*x_q),     1 - 2*(x_q*x_q + y_q*y_q),
        ], dim=-1).reshape(-1, 3, 3)  # (N, 3, 3)

        # Min-scale axis = surface normal direction
        min_axis = scales.argmin(dim=1)  # (N,)
        normals_world = torch.zeros(n_pts, 3, dtype=torch.float32, device=RENDER_DEVICE)
        for ax in range(3):
            mask = (min_axis == ax)
            if mask.any():
                normals_world[mask] = R_mat[mask, :, ax]

        # Transform to camera space
        R_cam = w2c[:3, :3]
        normals_cam = (R_cam @ normals_world.T).T  # (N, 3)

        # Flip normals facing away from camera
        flip = normals_cam[:, 2] > 0
        normals_cam[flip] *= -1

        # Normalize
        n_norm = normals_cam.norm(dim=1, keepdim=True).clamp(min=1e-8)
        normals_cam = normals_cam / n_norm

        # Encode to [0, 1] color range
        normal_colors = normals_cam * 0.5 + 0.5  # (N, 3) in [0, 1]

        normal_render = render_with_custom_colors(
            pc, height, width, c2w, fxfycxcy, normal_colors, bg_color=(0.5, 0.5, 0.5)
        )
        normal_map = normal_render.detach().cpu().numpy().transpose(1, 2, 0)  # (H, W, 3)
        normal_rgb = (normal_map.clip(0, 1) * 255).astype(np.uint8)

    # Foreground mask from opacity
    fg_mask = opacity_map > 0.5

    del pc
    torch.cuda.empty_cache()

    return {
        'rgb': rgb,
        'depth': depth_map,
        'normal': normal_rgb,
        'opacity': opacity_out,
        'mask': fg_mask,
    }


print('GPU rendering engine ready (ALL maps via CUDA 3DGS rasterizer)')
print(f'Device: {RENDER_DEVICE}')


## 1.2 Render every splat → RGB / depth / normal / opacity


In [ ]:
import shutil, time


FORCE_RERUN = False   # True = wipe + re-render everything; False = skip if count already matches

# --- Ensure output dirs exist (but do NOT wipe unless FORCE_RERUN) ---
for d in [DEPTH_DIR, NORMAL_DIR, OPACITY_DIR, RGB_DIR]:
    if FORCE_RERUN and d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
if FORCE_RERUN:
    print('[4.2] FORCE_RERUN=True \u2014 cleaned output directories')

# Rendering parameters from config

# Setup frontal camera (once)
fxfycxcy_gpu, c2w_gpu = get_frontal_camera(
    hfov=FOV, w=RENDER_RES, h=RENDER_RES, radius=CAM_DIST
)
print(f'Camera: FOV={FOV}, distance={CAM_DIST}, resolution={RENDER_RES}')

# Find all splat folders
splat_folders = sorted([
    d for d in SPLAT_DIR.iterdir()
    if d.is_dir() and (d / 'gaussians.ply').exists()
])
n_splats = len(splat_folders)
expected_stems = {d.name for d in splat_folders}
print(f'Total splats: {n_splats}')

# --- File-existence helpers ---------------------------------------------------
def _valid(p):
    """A rendered file counts as 'done' only if it exists AND is non-empty."""
    return p.exists() and p.stat().st_size > 0

def _stems(d):
    """Set of .png stems in a directory (ignores zero-byte files)."""
    return {p.stem for p in d.glob('*.png') if p.stat().st_size > 0}

def _count_png(d):
    return len(list(d.glob('*.png')))

counts = {
    'RGB':     _count_png(RGB_DIR),
    'Depth':   _count_png(DEPTH_DIR),
    'Normal':  _count_png(NORMAL_DIR),
    'Opacity': _count_png(OPACITY_DIR),
}

# --- Whole-batch skip: requires (a) counts all match, (b) stems match splats ---
count_match = all(c == n_splats for c in counts.values()) and n_splats > 0
stem_match  = False
if count_match:
    stems = {
        'RGB':     _stems(RGB_DIR),
        'Depth':   _stems(DEPTH_DIR),
        'Normal':  _stems(NORMAL_DIR),
        'Opacity': _stems(OPACITY_DIR),
    }
    stem_match = all(s == expected_stems for s in stems.values())
    if not stem_match:
        # Report mismatch in a short summary
        for k, s in stems.items():
            missing = expected_stems - s
            extra   = s - expected_stems
            if missing or extra:
                print(f'[4.2] {k} dir stem mismatch: missing={len(missing)} extra={len(extra)}')

if not FORCE_RERUN and count_match and stem_match:
    print(f'[4.2] SKIP \u2014 all 4 output dirs have {n_splats} files with matching stems. '
          f'Set FORCE_RERUN=True to re-render.')
    print(f'  RGB={counts["RGB"]}  Depth={counts["Depth"]}  '
          f'Normal={counts["Normal"]}  Opacity={counts["Opacity"]}')
else:
    if not FORCE_RERUN:
        print(f'[4.2] Current counts \u2014 RGB={counts["RGB"]}  Depth={counts["Depth"]}  '
              f'Normal={counts["Normal"]}  Opacity={counts["Opacity"]}  '
              f'(expected {n_splats}) \u2014 per-sample skip mode')

    t0 = time.time()
    n_done, n_skip = 0, 0

    for i, folder in enumerate(splat_folders):
        name = folder.name
        ply_path = folder / 'gaussians.ply'

        # Per-sample skip: all 4 output files must exist AND be non-empty
        targets = {
            'rgb':     RGB_DIR     / (name + '.png'),
            'depth':   DEPTH_DIR   / (name + '.png'),
            'normal':  NORMAL_DIR  / (name + '.png'),
            'opacity': OPACITY_DIR / (name + '.png'),
        }
        if not FORCE_RERUN and all(_valid(p) for p in targets.values()):
            n_skip += 1
            continue

        try:
            maps = render_all_maps_gpu(ply_path, fxfycxcy_gpu, c2w_gpu, RENDER_RES, RENDER_RES)

            Image.fromarray(maps['rgb']).save(targets['rgb'])

            depth_u16 = (maps['depth'].clip(0, 1) * 65535).astype(np.uint16)
            Image.fromarray(depth_u16, mode='I;16').save(targets['depth'])

            Image.fromarray(maps['normal']).save(targets['normal'])
            Image.fromarray(maps['opacity'], mode='L').save(targets['opacity'])
            n_done += 1

            if (n_done) % 10 == 0 or (i + 1) == n_splats:
                elapsed = time.time() - t0
                avg = elapsed / max(n_done, 1)
                remain = avg * (n_splats - i - 1)
                print(f'[{i+1}/{n_splats}] {name} | rendered={n_done} skipped={n_skip} | '
                      f'avg={avg:.1f}s | ETA={remain/60:.1f}min')

        except Exception as e:
            print(f'ERROR [{name}]: {e}')
            import traceback
            traceback.print_exc()

    elapsed = time.time() - t0
    print(f'\n[4.2] Done. Rendered {n_done}, skipped {n_skip} (total {n_splats}) '
          f'in {elapsed/60:.1f} min')
    print(f'  RGB:     {_count_png(RGB_DIR)}')
    print(f'  Depth:   {_count_png(DEPTH_DIR)}')
    print(f'  Normal:  {_count_png(NORMAL_DIR)}')
    print(f'  Opacity: {_count_png(OPACITY_DIR)}')

## 1.3 Postprocess refresh — re-scan `data/raw_faces`


In [ ]:

RAW_DIR = Path(config['paths']['dataset_raw']).resolve()
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
raw_images = sorted([f for f in RAW_DIR.iterdir() if f.suffix.lower() in image_exts])
print(f'Raw images currently in {RAW_DIR}: {len(raw_images)}')
for f in raw_images[:5]:
    print('  -', f.name)
if len(raw_images) > 5:
    print(f'  ... and {len(raw_images)-5} more')


## 2.1 Camera at arbitrary yaw


In [ ]:
def get_camera_at_yaw(yaw_deg=0.0, hfov=None, w=None, h=None, radius=None, elevation=0):
    """
    Return (fxfycxcy, c2w) tensors for a camera rotated by `yaw_deg` from frontal.
    Non-yaw params default to the config values used in Step 4.
    """
    if hfov   is None: hfov   = config['depth_export']['fov']
    if radius is None: radius = config['depth_export']['camera_distance']
    if w      is None: w      = config['depth_export']['render_resolution']
    if h      is None: h      = config['depth_export']['render_resolution']

    fx = w / (2 * np.tan(np.deg2rad(hfov) / 2.0))
    fy = fx
    cx, cy = w / 2.0, h / 2.0
    fxfycxcy = np.array([fx, fy, cx, cy], dtype=np.float32)

    elev = np.deg2rad(elevation)
    azim = np.deg2rad(270 + yaw_deg)  # 270 = frontal, offset by yaw
    z = radius * np.sin(elev)
    base = radius * np.cos(elev)
    x = base * np.cos(azim)
    y = base * np.sin(azim)
    cam_pos = np.array([x, y, z])

    up_vector = np.array([0, 0, 1])
    forward = -cam_pos / np.linalg.norm(cam_pos)
    right = np.cross(forward, up_vector)
    right = right / np.linalg.norm(right)
    up = np.cross(right, forward)
    up = up / np.linalg.norm(up)

    R = np.stack((right, -up, forward), axis=1)
    c2w = np.eye(4, dtype=np.float32)
    c2w[:3, :4] = np.concatenate((R, cam_pos[:, None]), axis=1)

    fxfycxcy_t = torch.from_numpy(fxfycxcy).float().to(RENDER_DEVICE)
    c2w_t      = torch.from_numpy(c2w).float().to(RENDER_DEVICE)
    return fxfycxcy_t, c2w_t


print('get_camera_at_yaw ready (yaw_deg offsets azimuth from frontal 270°)')

## 2.2 Multi-view render with shared global depth normalization


In [ ]:

MULTIVIEW_DIR.mkdir(parents=True, exist_ok=True)

splat_folders = sorted([d for d in SPLAT_DIR.iterdir()
                        if d.is_dir() and (d / 'gaussians.ply').exists()])
assert len(splat_folders) > 0, 'No splats found — run Step 3 first.'

SAMPLE_IDX = 0
sample     = splat_folders[SAMPLE_IDX]
ply_path   = sample / 'gaussians.ply'
print(f'Sample [{SAMPLE_IDX}]: {sample.name}')

yaw_list = [0, 15]
RES      = config['depth_export']['render_resolution']

# Build all cameras first
cams = [(yaw, *get_camera_at_yaw(yaw_deg=yaw)) for yaw in yaw_list]

# Load pc once
pc = GS_GaussianModel(sh_degree=3)
pc.load_ply(str(ply_path))
pc = pc.to(RENDER_DEVICE)
n_pts = pc.get_xyz.shape[0]

# ---- PASS 1: compute GLOBAL depth range across all views ----
with torch.no_grad():
    d_min_g, d_max_g = float('inf'), float('-inf')
    for yaw, fxfycxcy, c2w in cams:
        vc = Camera(C2W=c2w, fxfycxcy=fxfycxcy, h=RES, w=RES)
        w2c = vc.world_view_transform.T
        xyz_h  = torch.cat([pc.get_xyz, torch.ones(n_pts, 1, device=RENDER_DEVICE)], dim=1)
        xyz_cam = (w2c @ xyz_h.T).T[:, :3]
        depths  = xyz_cam[:, 2]
        valid   = depths > 0.1
        if valid.any():
            d_min_g = min(d_min_g, float(depths[valid].min()))
            d_max_g = max(d_max_g, float(depths[valid].max()))
    print(f'Global depth range across {len(cams)} views: [{d_min_g:.4f}, {d_max_g:.4f}]')
    d_range = max(d_max_g - d_min_g, 1e-8)

# ---- PASS 2: render each view using the SHARED global range ----
results = {}
with torch.no_grad():
    for yaw, fxfycxcy, c2w in cams:
        # RGB
        rgb = render_opencv_cam(pc, RES, RES, c2w, fxfycxcy, bg_color=(1.0, 1.0, 1.0))['render']
        rgb = (rgb.detach().cpu().numpy() * 255).clip(0, 255).astype(np.uint8).transpose(1, 2, 0)

        # Opacity
        white = torch.ones(n_pts, 3, dtype=torch.float32, device=RENDER_DEVICE)
        op_r  = render_with_custom_colors(pc, RES, RES, c2w, fxfycxcy, white, bg_color=(0.0, 0.0, 0.0))
        op    = (op_r[0].detach().cpu().numpy().clip(0, 1) * 255).astype(np.uint8)

        # Depth with GLOBAL normalization
        vc = Camera(C2W=c2w, fxfycxcy=fxfycxcy, h=RES, w=RES)
        w2c = vc.world_view_transform.T
        xyz_h  = torch.cat([pc.get_xyz, torch.ones(n_pts, 1, device=RENDER_DEVICE)], dim=1)
        xyz_cam = (w2c @ xyz_h.T).T[:, :3]
        depths  = xyz_cam[:, 2]
        depth_norm = (1.0 - (depths - d_min_g) / d_range).clamp(0, 1)  # closer = brighter
        d_colors = depth_norm.unsqueeze(1).expand(-1, 3)
        d_r = render_with_custom_colors(pc, RES, RES, c2w, fxfycxcy, d_colors, bg_color=(0.0, 0.0, 0.0))
        depth_map = d_r[0].detach().cpu().numpy().clip(0, 1)

        results[yaw] = {'rgb': rgb, 'depth': depth_map, 'opacity': op}

        out_dir = MULTIVIEW_DIR / sample.name / f'yaw_{yaw:+d}'
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(rgb).save(out_dir / 'rgb.png')
        Image.fromarray((depth_map * 65535).astype(np.uint16), mode='I;16').save(out_dir / 'depth.png')
        Image.fromarray(op, mode='L').save(out_dir / 'opacity.png')
        print(f'  yaw={yaw:+d}° saved')

    (MULTIVIEW_DIR / sample.name / 'depth_range.txt').write_text(
        f'd_min={d_min_g:.6f}\nd_max={d_max_g:.6f}\nyaws={yaw_list}\n'
    )

del pc
torch.cuda.empty_cache()

# Display (same global vmin/vmax → visually comparable)
fig, axes = plt.subplots(2, len(yaw_list), figsize=(4 * len(yaw_list), 8))
for j, yaw in enumerate(yaw_list):
    axes[0, j].imshow(results[yaw]['rgb'])
    axes[0, j].set_title(f'RGB  |  yaw={yaw:+d}°')
    axes[0, j].axis('off')
    axes[1, j].imshow(results[yaw]['depth'], cmap='magma', vmin=0, vmax=1)
    axes[1, j].set_title(f'Depth (global norm)  |  yaw={yaw:+d}°')
    axes[1, j].axis('off')
plt.tight_layout()
plt.show()

print(f'\nGlobal depth range: [{d_min_g:.4f}, {d_max_g:.4f}]')
print(f'Saved to: {MULTIVIEW_DIR / sample.name}')

## 2.3 Reprojection consistency via tight foreground mask


In [ ]:
import cv2
from scipy.ndimage import binary_erosion

OPACITY_TAU      = 0.9    # tight: only solid foreground
ERODE_PX         = 5      # morphological erosion radius in pixels
DEPTH_TOL_FRAC   = 0.02   # |delta Z| tolerance as fraction of global depth range
MAX_BASELINE_DEG = 20     # skip pairs with yaw baseline larger than this (near-frontal only)

# ---- Face mask from cropped_faces (pre-FaceLift source) ----
# Section 3.1 builds & saves these masks at data/dataset/{train,val}/mask/<stem>.png.
# For the §5 consistency pass the splat name may not match a dataset split stem
# yet, so we also support building on-the-fly from data/cropped_faces/<name>.png.
CROPPED_FACES_DIR = Path(config['paths']['dataset_raw']).parent / 'cropped_faces'
# Mask construction params — tightened to kill hair-edge / matting grey halo:
#   WHITE_TOL was 12, way too loose — grey matting edges (distance ~15-20) were
#     being classified as foreground. Bumped to 30.
#   FACE_MASK_ERODE_PX pulls the final mask in by 5 px so we never sit on the
#     hair boundary.
FACE_MASK_WHITE_TOL = 30
FACE_MASK_K_CLOSE   = 7
FACE_MASK_K_OPEN    = 3
FACE_MASK_ERODE_PX  = 5


def _build_face_mask_from_cropped(
    name, res=1024,
    white_tol=None, k_close=None, k_open=None, erode_px=None,
    return_rgb=False,
):
    """Load data/cropped_faces/<name>.png and return a binary (res,res) face mask.

    Single source of truth for the face mask. Used by:
      - sections 2.3 / 2.4 / 2.5 (consistency, A-side fg mask for reproject_consistency)
      - section 5.2 (preview worst samples)
      - section 3.1 (dataset mask generation — called with loose params)

    Build mask at SOURCE resolution (no RGB resize — that softens edges) ->
    morphology -> largest component -> erode -> resize with NEAREST -> re-binarize.

    Parameters default to the FACE_MASK_* globals (tight: WHITE_TOL=30, erode=5).
    Callers that want a different profile (e.g. 6.1 dataset mask uses a looser
    WHITE_TOL=12, no erode) pass explicit kwargs.

    Returns:
        mask (H,W) bool, or None if the source image is missing.
        If return_rgb=True, returns (rgb (H,W,3) uint8, mask (H,W) bool) instead.
    """
    # Resolve params (fall back to globals)
    wt = FACE_MASK_WHITE_TOL if white_tol is None else white_tol
    kc = FACE_MASK_K_CLOSE   if k_close   is None else k_close
    ko = FACE_MASK_K_OPEN    if k_open    is None else k_open
    ep = FACE_MASK_ERODE_PX  if erode_px  is None else erode_px

    src = CROPPED_FACES_DIR / f"{name}.png"
    if not src.exists():
        for ext in ('.jpg', '.jpeg', '.webp', '.bmp'):
            alt = CROPPED_FACES_DIR / f"{name}{ext}"
            if alt.exists():
                src = alt
                break
        else:
            return (None, None) if return_rgb else None

    rgb = np.array(Image.open(src).convert("RGB"))

    # Build mask at SOURCE resolution (never resize the RGB before thresholding)
    dist = (255 - rgb.astype(np.int16)).max(axis=-1)
    m = (dist > wt).astype(np.uint8) * 255

    if kc > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kc, kc))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    if ko > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ko, ko))
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats((m > 0).astype(np.uint8), 8)
    if n_labels > 1:
        best = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        m = np.where(labels == best, 255, 0).astype(np.uint8)

    # Fill any interior holes inside the face blob (eye sclera / teeth / specular
    # patches can get classified as background by the white-distance threshold,
    # leaving NaN dots in masked-depth displays). Flood-fill from a border pixel
    # to mark the true exterior, then anything still 0 is an interior hole.
    h_m, w_m = m.shape
    flood = m.copy()
    ff_mask = np.zeros((h_m + 2, w_m + 2), np.uint8)
    cv2.floodFill(flood, ff_mask, (0, 0), 255)
    m[flood == 0] = 255

    if ep > 0:
        k = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * ep + 1, 2 * ep + 1),
        )
        m = cv2.erode(m, k, iterations=1)

    # Resize mask with NEAREST, re-binarize — never bilinear on a binary mask
    if m.shape[0] != res or m.shape[1] != res:
        m = cv2.resize(m, (res, res), interpolation=cv2.INTER_NEAREST)
    mask_out = m > 127

    if return_rgb:
        if rgb.shape[0] != res or rgb.shape[1] != res:
            rgb_out = cv2.resize(rgb, (res, res), interpolation=cv2.INTER_AREA)
        else:
            rgb_out = rgb
        return rgb_out, mask_out
    return mask_out


def _invert_depth_norm(depth_norm_np, d_min, d_max):
    """depth_norm = 1 - (z - d_min)/(d_max-d_min)  =>  z = d_max - depth_norm*(d_max-d_min)."""
    d_range = max(d_max - d_min, 1e-8)
    return d_max - depth_norm_np.astype(np.float32) * d_range


def _build_intrinsics(fxfycxcy_t):
    fx, fy, cx, cy = fxfycxcy_t.detach().cpu().numpy().tolist()
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float32)


def _tight_fg_mask(opacity_u8, tau=OPACITY_TAU, erode_px=ERODE_PX):
    m = (opacity_u8.astype(np.float32) / 255.0) > tau
    if erode_px > 0:
        struct = np.ones((2 * erode_px + 1, 2 * erode_px + 1), dtype=bool)
        m = binary_erosion(m, structure=struct, iterations=1)
    return m


def reproject_consistency(A, B, d_min, d_max,
                          face_mask_A=None,
                          opacity_tau=OPACITY_TAU, erode_px=ERODE_PX,
                          depth_tol_frac=DEPTH_TOL_FRAC,
                          max_baseline_deg=MAX_BASELINE_DEG):
    """
    A, B: dicts with keys 'yaw','fxfycxcy','c2w','depth'(normalized),'opacity'(u8).
    face_mask_A: optional bool (H,W) face mask in the A view (from cropped_faces).
                 When provided, A-side foreground is restricted to face_mask_A,
                 i.e. consistency is only evaluated inside the person region from
                 the pre-FaceLift source image — this prevents hallucinated neck /
                 shoulder geometry from contaminating the metric.
    Returns dict of results, or None if baseline > max_baseline_deg.
    """
    baseline = abs(A['yaw'] - B['yaw'])
    if baseline > max_baseline_deg:
        return None

    H, W = A['depth'].shape
    KA = _build_intrinsics(A['fxfycxcy'])
    KB = _build_intrinsics(B['fxfycxcy'])
    c2w_A = A['c2w'].detach().cpu().numpy()
    c2w_B = B['c2w'].detach().cpu().numpy()
    w2c_B = np.linalg.inv(c2w_B)
    d_range = max(d_max - d_min, 1e-8)

    # Invert A's depth back to camera-space Z
    z_A = _invert_depth_norm(A['depth'], d_min, d_max)

    # TIGHTENED foreground mask in A:
    #   opacity-based tight mask  ∩  cropped_faces face mask (if given)
    fg_A = _tight_fg_mask(A['opacity'], opacity_tau, erode_px) & (z_A > 0.1)
    if face_mask_A is not None:
        if face_mask_A.shape != fg_A.shape:
            face_mask_A = cv2.resize(
                face_mask_A.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST
            ).astype(bool)
        fg_A = fg_A & face_mask_A

    us, vs = np.meshgrid(np.arange(W), np.arange(H))
    us = us.astype(np.float32); vs = vs.astype(np.float32)

    # Backproject A pixels -> camera-A coords
    x_cam = (us - KA[0, 2]) * z_A / KA[0, 0]
    y_cam = (vs - KA[1, 2]) * z_A / KA[1, 1]
    pts_camA = np.stack([x_cam, y_cam, z_A], axis=-1)
    pts_camA_h = np.concatenate([pts_camA, np.ones_like(z_A)[..., None]], axis=-1)

    # camera-A -> world -> camera-B
    pts_world  = pts_camA_h @ c2w_A.T
    pts_camB_h = pts_world @ w2c_B.T
    X, Y, Z = pts_camB_h[..., 0], pts_camB_h[..., 1], pts_camB_h[..., 2]
    valid_front = Z > 0.1

    # Project into B
    uB = KB[0, 0] * X / np.maximum(Z, 1e-8) + KB[0, 2]
    vB = KB[1, 1] * Y / np.maximum(Z, 1e-8) + KB[1, 2]
    in_bounds = (uB >= 0) & (uB < W - 1) & (vB >= 0) & (vB < H - 1)

    # Bilinear sampler
    uB_c = np.clip(uB, 0, W - 1); vB_c = np.clip(vB, 0, H - 1)
    u0 = np.floor(uB_c).astype(np.int32); u1 = np.minimum(u0 + 1, W - 1)
    v0 = np.floor(vB_c).astype(np.int32); v1 = np.minimum(v0 + 1, H - 1)
    du = uB_c - u0; dv = vB_c - v0
    def _bilerp(img):
        return (img[v0, u0] * (1 - du) * (1 - dv) +
                img[v0, u1] * du       * (1 - dv) +
                img[v1, u0] * (1 - du) * dv       +
                img[v1, u1] * du       * dv)

    # Sample B's depth at reprojected pixels and invert to world units
    depthB_sampled_norm = _bilerp(B['depth'])
    z_B_sampled = _invert_depth_norm(depthB_sampled_norm, d_min, d_max)

    # B-side stays as opacity tight mask — person has rotated, cropped_faces mask
    # would no longer align. A-side face restriction is enough: we only reproject
    # A face pixels, so we only need B to confirm they land on solid geometry.
    fgB_bin = _tight_fg_mask(B['opacity'], opacity_tau, erode_px).astype(np.float32)
    fg_B = _bilerp(fgB_bin) > 0.99

    # Depth consistency
    dz = np.abs(Z - z_B_sampled)
    depth_tol = depth_tol_frac * d_range
    consistent = dz < depth_tol

    visibility = fg_A & valid_front & in_bounds & fg_B
    final_mask = visibility & consistent

    total_fg  = int(fg_A.sum())
    total_vis = int(visibility.sum())
    total_ok  = int(final_mask.sum())
    cons_rate = total_ok / max(total_vis, 1)
    mean_dz   = float(dz[visibility].mean()) if total_vis > 0 else float('nan')

    return {
        'visibility': visibility, 'consistent': final_mask, 'dz': dz,
        'cons_rate': cons_rate, 'mean_dz': mean_dz,
        'total_fg': total_fg, 'total_vis': total_vis, 'total_ok': total_ok,
        'depth_tol': depth_tol, 'baseline_deg': baseline,
    }


# Package camera + render outputs per view for the checker
views = {}
for (yaw, fxfycxcy, c2w) in cams:
    views[yaw] = {
        'yaw': yaw, 'fxfycxcy': fxfycxcy, 'c2w': c2w,
        'depth': results[yaw]['depth'], 'opacity': results[yaw]['opacity'],
    }

# Face mask for the A view (yaw=0) comes from cropped_faces — the pre-FaceLift source
face_mask_A = _build_face_mask_from_cropped(sample.name, res=views[yaw_list[0]]['depth'].shape[0])
if face_mask_A is None:
    print(f'[WARN] no cropped_faces source for {sample.name} — falling back to opacity-only mask')

# Run both directions (single-sample demo)
pair_results = {}
for a, b in [(yaw_list[0], yaw_list[1]), (yaw_list[1], yaw_list[0])]:
    # Only the A=yaw_list[0] side has an aligned cropped_faces mask (person is
    # frontal). For the reverse direction, omit face_mask_A so the metric is not
    # misaligned.
    fm = face_mask_A if a == yaw_list[0] else None
    r = reproject_consistency(views[a], views[b], d_min_g, d_max_g, face_mask_A=fm)
    if r is None:
        print(f'[{a:+d}° -> {b:+d}°]  skipped (baseline > {MAX_BASELINE_DEG}°)')
        continue
    pair_results[(a, b)] = r
    print(f'[{a:+d}° -> {b:+d}°]  baseline={r["baseline_deg"]}°  '
          f'cons_rate={r["cons_rate"]*100:5.2f}%  '
          f'mean|dz|={r["mean_dz"]:.4f} (tol={r["depth_tol"]:.4f})  '
          f'fg={r["total_fg"]}  vis={r["total_vis"]}  ok={r["total_ok"]}')

# Save per-pair masks + dz maps
(MULTIVIEW_DIR / sample.name / 'consistency').mkdir(parents=True, exist_ok=True)
for (a, b), r in pair_results.items():
    tag = f'{a:+d}_to_{b:+d}'.replace('+', 'p').replace('-', 'm')
    np.savez_compressed(
        MULTIVIEW_DIR / sample.name / 'consistency' / f'{tag}.npz',
        visibility=r['visibility'], consistent=r['consistent'], dz=r['dz'],
        cons_rate=r['cons_rate'], mean_dz=r['mean_dz'], depth_tol=r['depth_tol'],
    )

# Visualize the first pair
if pair_results:
    a, b = yaw_list[0], yaw_list[1]
    r = pair_results[(a, b)]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(views[a]['depth'], cmap='magma', vmin=0, vmax=1)
    axes[0].set_title(f'Depth A (yaw={a:+d}°)'); axes[0].axis('off')
    axes[1].imshow(r['visibility'], cmap='gray')
    axes[1].set_title(f'Tight visibility ({r["total_vis"]} px)'); axes[1].axis('off')
    axes[2].imshow(r['consistent'], cmap='gray')
    axes[2].set_title(f'Consistent (rate={r["cons_rate"]*100:.1f}%)'); axes[2].axis('off')
    im = axes[3].imshow(np.where(r['visibility'], r['dz'], np.nan), cmap='inferno')
    axes[3].set_title('|Δz| (world units)'); axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046)
    plt.tight_layout()
    plt.show()

print(f'\nConsistency artifacts saved to: {MULTIVIEW_DIR / sample.name / "consistency"}')

## 2.4 Batch consistency over all splats


In [ ]:

BATCH_YAWS  = [0, 15]   # small baseline — near-frontal only
MAX_SAMPLES = None      # None = all splats; set to an int to subsample for a quick pass
FORCE_RERUN = False     # True = always re-run batch; False = reuse consistency_summary.csv if row count matches

splat_folders_all = sorted([d for d in SPLAT_DIR.iterdir()
                            if d.is_dir() and (d / 'gaussians.ply').exists()])
if MAX_SAMPLES is not None:
    splat_folders_all = splat_folders_all[:MAX_SAMPLES]
print(f'[5.4] Batch consistency on {len(splat_folders_all)} splats, yaws={BATCH_YAWS}')
print(f'      Face masks sourced from: {CROPPED_FACES_DIR}')

summary_csv = MULTIVIEW_DIR / 'consistency_summary.csv'

# --- Skip-if-exists: if CSV already exists with row count == n_splats, just load it ---
rows = None
if not FORCE_RERUN and summary_csv.exists():
    with open(summary_csv, 'r', newline='') as f:
        reader = csv.DictReader(f)
        cached = list(reader)
    if len(cached) == len(splat_folders_all):
        print(f'[5.4] SKIP — {summary_csv.name} already has {len(cached)} rows '
              f'(matches splat count). Set FORCE_RERUN=True to re-compute.')
        rows = []
        for r in cached:
            rows.append({
                'name':          r['name'],
                'cons_rate':     float(r['cons_rate']),
                'mean_dz':       float(r['mean_dz']),
                'n_vis':         int(float(r['n_vis'])),
                'd_min':         float(r['d_min']),
                'd_max':         float(r['d_max']),
                'has_face_mask': int(r.get('has_face_mask', 0)),
            })
    else:
        print(f'[5.4] Cache row count mismatch ({len(cached)} vs {len(splat_folders_all)}) '
              f'— re-running batch.')


def _render_views_shared_range(ply_path, yaws, res):
    """Load pc once, render all `yaws` with a SHARED (d_min,d_max) across views."""
    pc = GS_GaussianModel(sh_degree=3)
    pc.load_ply(str(ply_path))
    pc = pc.to(RENDER_DEVICE)
    n_pts = pc.get_xyz.shape[0]
    cam_list = [(y, *get_camera_at_yaw(yaw_deg=y)) for y in yaws]

    with torch.no_grad():
        # Global range across all requested views
        d_min, d_max = float('inf'), float('-inf')
        for y, fx, c2w in cam_list:
            vc = Camera(C2W=c2w, fxfycxcy=fx, h=res, w=res)
            w2c = vc.world_view_transform.T
            xyz_h = torch.cat([pc.get_xyz, torch.ones(n_pts, 1, device=RENDER_DEVICE)], dim=1)
            zs = (w2c @ xyz_h.T).T[:, 2]
            v = zs > 0.1
            if v.any():
                d_min = min(d_min, float(zs[v].min()))
                d_max = max(d_max, float(zs[v].max()))
        d_rng = max(d_max - d_min, 1e-8)

        out = {}
        for y, fx, c2w in cam_list:
            # Opacity render
            white = torch.ones(n_pts, 3, dtype=torch.float32, device=RENDER_DEVICE)
            op_r = render_with_custom_colors(pc, res, res, c2w, fx, white, bg_color=(0., 0., 0.))
            op = (op_r[0].detach().cpu().numpy().clip(0, 1) * 255).astype(np.uint8)

            # Depth render with shared global norm
            vc = Camera(C2W=c2w, fxfycxcy=fx, h=res, w=res)
            w2c = vc.world_view_transform.T
            xyz_h = torch.cat([pc.get_xyz, torch.ones(n_pts, 1, device=RENDER_DEVICE)], dim=1)
            zs = (w2c @ xyz_h.T).T[:, 2]
            dn = (1.0 - (zs - d_min) / d_rng).clamp(0, 1)
            d_col = dn.unsqueeze(1).expand(-1, 3)
            d_r = render_with_custom_colors(pc, res, res, c2w, fx, d_col, bg_color=(0., 0., 0.))
            depth_map = d_r[0].detach().cpu().numpy().clip(0, 1)

            out[y] = {
                'yaw': y, 'fxfycxcy': fx, 'c2w': c2w,
                'depth': depth_map, 'opacity': op,
            }

    del pc
    torch.cuda.empty_cache()
    return out, d_min, d_max


# --- Run batch (only if cache didn't hit) ---
if rows is None:
    rows = []
    n_no_face_mask = 0
    t0 = time.time()
    for i, folder in enumerate(splat_folders_all):
        name = folder.name
        ply_path = folder / 'gaussians.ply'
        try:
            vs, d_min, d_max = _render_views_shared_range(ply_path, BATCH_YAWS, RES)

            # Build face mask for the A view (aligned to yaw=0 from cropped_faces)
            face_mask = _build_face_mask_from_cropped(name, res=RES)
            if face_mask is None:
                n_no_face_mask += 1

            # Both directions, averaged. Only the A=BATCH_YAWS[0] direction gets
            # the face mask (yaw=0 is the aligned frame). Reverse direction stays
            # opacity-only since the person has rotated by BATCH_YAWS[1].
            rates, dzs, n_viss = [], [], []
            for a, b in [(BATCH_YAWS[0], BATCH_YAWS[1]), (BATCH_YAWS[1], BATCH_YAWS[0])]:
                fm = face_mask if a == BATCH_YAWS[0] else None
                r = reproject_consistency(vs[a], vs[b], d_min, d_max, face_mask_A=fm)
                if r is None or r['total_vis'] == 0:
                    continue
                rates.append(r['cons_rate'])
                dzs.append(r['mean_dz'])
                n_viss.append(r['total_vis'])

            if not rates:
                continue

            rows.append({
                'name':      name,
                'cons_rate': float(np.mean(rates)),
                'mean_dz':   float(np.nanmean(dzs)),
                'n_vis':     int(np.mean(n_viss)),
                'd_min':     d_min,
                'd_max':     d_max,
                'has_face_mask': int(face_mask is not None),
            })
        except Exception as e:
            print(f'ERROR [{name}]: {e}')
            continue

        if (i + 1) % 10 == 0 or (i + 1) == len(splat_folders_all):
            elapsed = time.time() - t0
            avg = elapsed / (i + 1)
            remain = avg * (len(splat_folders_all) - i - 1)
            print(f'  [{i+1}/{len(splat_folders_all)}] avg={avg:.2f}s/sample  ETA={remain/60:.1f}min')

    print(f'\nBatch done. Successful samples: {len(rows)} / {len(splat_folders_all)}')
    if n_no_face_mask > 0:
        print(f'[WARN] {n_no_face_mask} samples had no cropped_faces source — opacity-only fallback')

    # --- Save CSV ---
    with open(summary_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['name', 'cons_rate', 'mean_dz', 'n_vis',
                                           'd_min', 'd_max', 'has_face_mask'])
        w.writeheader()
        w.writerows(rows)
    print(f'Summary CSV: {summary_csv}')

# --- Aggregate + visualize ---
if rows:
    cons_arr = np.array([r['cons_rate'] for r in rows])
    dz_arr   = np.array([r['mean_dz']   for r in rows])
    nv_arr   = np.array([r['n_vis']     for r in rows])

    print(f'\ncons_rate:  mean={cons_arr.mean()*100:6.2f}%  '
          f'median={np.median(cons_arr)*100:6.2f}%  '
          f'std={cons_arr.std()*100:5.2f}%  '
          f'min={cons_arr.min()*100:6.2f}%  max={cons_arr.max()*100:6.2f}%')
    print(f'mean|Δz|:   mean={dz_arr.mean():.4f}  median={np.median(dz_arr):.4f}  '
          f'max={dz_arr.max():.4f}')
    print(f'n_vis:      mean={nv_arr.mean():.0f}  median={np.median(nv_arr):.0f}')

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(cons_arr * 100, bins=30, color='steelblue', edgecolor='k')
    axes[0].axvline(cons_arr.mean() * 100, color='r', ls='--',
                    label=f'mean={cons_arr.mean()*100:.1f}%')
    axes[0].set_xlabel('Consistency rate (%)')
    axes[0].set_ylabel('# samples')
    axes[0].set_title(f'Per-sample cons_rate (n={len(rows)})')
    axes[0].legend()

    axes[1].hist(dz_arr, bins=30, color='seagreen', edgecolor='k')
    axes[1].axvline(dz_arr.mean(), color='r', ls='--',
                    label=f'mean={dz_arr.mean():.4f}')
    axes[1].set_xlabel('Mean |Δz| (world units)')
    axes[1].set_ylabel('# samples')
    axes[1].set_title('Per-sample mean|Δz|')
    axes[1].legend()

    axes[2].scatter(nv_arr, cons_arr * 100, s=12, alpha=0.6)
    axes[2].set_xlabel('# visible pixels (after tight mask)')
    axes[2].set_ylabel('cons_rate (%)')
    axes[2].set_title('Visibility vs consistency')
    plt.tight_layout()
    plt.show()

    # --- Qualitative: top-3 best / worst (re-render on demand) ---
    sorted_rows = sorted(rows, key=lambda r: r['cons_rate'])
    worst = sorted_rows[:3]
    best  = sorted_rows[-3:][::-1]

    def _render_rgb(name):
        ply = SPLAT_DIR / name / 'gaussians.ply'
        pc = GS_GaussianModel(sh_degree=3)
        pc.load_ply(str(ply))
        pc = pc.to(RENDER_DEVICE)
        with torch.no_grad():
            fx, c2w = get_camera_at_yaw(yaw_deg=BATCH_YAWS[0])
            rgb = render_opencv_cam(pc, RES, RES, c2w, fx, bg_color=(1., 1., 1.))['render']
            rgb = (rgb.detach().cpu().numpy() * 255).clip(0, 255).astype(np.uint8).transpose(1, 2, 0)
        del pc
        torch.cuda.empty_cache()
        return rgb

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for col, r in enumerate(best):
        axes[0, col].imshow(_render_rgb(r['name']))
        axes[0, col].set_title(f"BEST  {r['cons_rate']*100:.1f}%\n{r['name'][:24]}")
        axes[0, col].axis('off')
    for col, r in enumerate(worst):
        axes[1, col].imshow(_render_rgb(r['name']))
        axes[1, col].set_title(f"WORST {r['cons_rate']*100:.1f}%\n{r['name'][:24]}")
        axes[1, col].axis('off')
    plt.tight_layout()
    plt.show()


## 2.5 Dual batch — cons_rate before vs after face mask


In [ ]:

# Reuse BATCH_YAWS / MAX_SAMPLES / SPLAT_DIR / RES / MULTIVIEW_DIR from section 2.4.
# Helpers needed (defined in previous cells):
#   _render_views_shared_range, _build_face_mask_from_cropped, reproject_consistency
BATCH_YAWS  = [0, 15]
MAX_SAMPLES = None       # None = all splats; set to an int to quick-check
FORCE_RERUN = False      # True = always re-run the batch; False = reuse cached CSV if row count matches

dual_csv    = MULTIVIEW_DIR / 'consistency_summary_both.csv'
summary_csv = MULTIVIEW_DIR / 'consistency_summary.csv'   # compat output for sections 2.4 / 2.6 / 5.0 / 5.2

# Need the splat count for both the skip check and batch loop
splat_folders_all = sorted([d for d in SPLAT_DIR.iterdir()
                            if d.is_dir() and (d / 'gaussians.ply').exists()])
if MAX_SAMPLES is not None:
    splat_folders_all = splat_folders_all[:MAX_SAMPLES]
n_splats = len(splat_folders_all)

rows = None

# --- Skip-if-exists: row count must match n_splats ---
if dual_csv.exists() and not FORCE_RERUN:
    with open(dual_csv, 'r', newline='') as f:
        cached = list(csv.DictReader(f))
    if len(cached) == n_splats:
        print(f'[5.4b] SKIP — {dual_csv.name} already has {len(cached)} rows '
              f'(matches splat count). Set FORCE_RERUN=True to re-compute.')
        rows = []
        for r in cached:
            rows.append({
                'name':           r['name'],
                'cons_rate_raw':  float(r['cons_rate_raw']),
                'cons_rate_mask': float(r['cons_rate_mask']),
                'mean_dz_raw':    float(r['mean_dz_raw']),
                'mean_dz_mask':   float(r['mean_dz_mask']),
                'n_vis_raw':      int(float(r['n_vis_raw'])),
                'n_vis_mask':     int(float(r['n_vis_mask'])),
                'd_min':          float(r['d_min']),
                'd_max':          float(r['d_max']),
                'has_face_mask':  int(r['has_face_mask']),
            })
    else:
        print(f'[5.4b] Cache row count mismatch ({len(cached)} vs {n_splats}) — re-running batch.')

# --- Fresh batch path: re-render + re-compute if no cache (or forced or mismatch) ---
if rows is None:
    print(f'[5.4b] Batch on {n_splats} splats, yaws={BATCH_YAWS}')

    rows = []
    n_no_face_mask = 0
    t0 = time.time()
    for i, folder in enumerate(splat_folders_all):
        name = folder.name
        ply_path = folder / 'gaussians.ply'
        try:
            vs, d_min, d_max = _render_views_shared_range(ply_path, BATCH_YAWS, RES)

            face_mask = _build_face_mask_from_cropped(name, res=RES)
            if face_mask is None:
                n_no_face_mask += 1

            # Per direction, compute BOTH variants: raw (no face mask) and masked.
            # Reverse direction (B -> A) stays opacity-only in both variants, because
            # the person has rotated by BATCH_YAWS[1] and the cropped_faces mask only
            # aligns to the yaw=0 (A) frame. So "raw" vs "masked" only differs on the
            # A -> B direction.
            rates_raw,  dzs_raw,  nvis_raw  = [], [], []
            rates_mask, dzs_mask, nvis_mask = [], [], []

            for a, b in [(BATCH_YAWS[0], BATCH_YAWS[1]), (BATCH_YAWS[1], BATCH_YAWS[0])]:
                r_raw = reproject_consistency(vs[a], vs[b], d_min, d_max, face_mask_A=None)
                if r_raw is not None and r_raw['total_vis'] > 0:
                    rates_raw.append(r_raw['cons_rate'])
                    dzs_raw.append(r_raw['mean_dz'])
                    nvis_raw.append(r_raw['total_vis'])

                fm = face_mask if a == BATCH_YAWS[0] else None
                r_m = reproject_consistency(vs[a], vs[b], d_min, d_max, face_mask_A=fm)
                if r_m is not None and r_m['total_vis'] > 0:
                    rates_mask.append(r_m['cons_rate'])
                    dzs_mask.append(r_m['mean_dz'])
                    nvis_mask.append(r_m['total_vis'])

            if not rates_raw or not rates_mask:
                continue

            rows.append({
                'name':           name,
                'cons_rate_raw':  float(np.mean(rates_raw)),
                'cons_rate_mask': float(np.mean(rates_mask)),
                'mean_dz_raw':    float(np.nanmean(dzs_raw)),
                'mean_dz_mask':   float(np.nanmean(dzs_mask)),
                'n_vis_raw':      int(np.mean(nvis_raw)),
                'n_vis_mask':     int(np.mean(nvis_mask)),
                'd_min':          d_min,
                'd_max':          d_max,
                'has_face_mask':  int(face_mask is not None),
            })
        except Exception as e:
            print(f'ERROR [{name}]: {e}')
            continue

        if (i + 1) % 10 == 0 or (i + 1) == n_splats:
            elapsed = time.time() - t0
            avg = elapsed / (i + 1)
            remain = avg * (n_splats - i - 1)
            print(f'  [{i+1}/{n_splats}] avg={avg:.2f}s/sample  ETA={remain/60:.1f}min')

    print(f'\n[5.4b] Successful samples: {len(rows)} / {n_splats}')
    if n_no_face_mask > 0:
        print(f'[WARN] {n_no_face_mask} samples had no cropped_faces source')

    # --- Save dual CSV ---
    dual_fields = ['name', 'cons_rate_raw', 'cons_rate_mask',
                   'mean_dz_raw', 'mean_dz_mask',
                   'n_vis_raw', 'n_vis_mask',
                   'd_min', 'd_max', 'has_face_mask']
    with open(dual_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=dual_fields)
        w.writeheader()
        w.writerows(rows)
    print(f'Dual CSV: {dual_csv}')

    # --- Also write the single-variant consistency_summary.csv (section 2.4 compat) ---
    # cons_rate / mean_dz / n_vis come from the MASKED variant (matches section 2.4's semantics)
    single_fields = ['name', 'cons_rate', 'mean_dz', 'n_vis', 'd_min', 'd_max', 'has_face_mask']
    with open(summary_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=single_fields)
        w.writeheader()
        for r in rows:
            w.writerow({
                'name':          r['name'],
                'cons_rate':     r['cons_rate_mask'],
                'mean_dz':       r['mean_dz_mask'],
                'n_vis':         r['n_vis_mask'],
                'd_min':         r['d_min'],
                'd_max':         r['d_max'],
                'has_face_mask': r['has_face_mask'],
            })
    print(f'Compat CSV: {summary_csv}  (sections 2.4 / 2.6 / 5.0 / 5.2 read this one)')

# --- BEFORE / AFTER comparison plot (2 rows x 3 cols) ---
if rows:
    cons_raw  = np.array([r['cons_rate_raw']  for r in rows])
    cons_mask = np.array([r['cons_rate_mask'] for r in rows])
    dz_raw    = np.array([r['mean_dz_raw']    for r in rows])
    dz_mask   = np.array([r['mean_dz_mask']   for r in rows])
    nv_raw    = np.array([r['n_vis_raw']      for r in rows])
    nv_mask   = np.array([r['n_vis_mask']     for r in rows])

    cons_min = min(cons_raw.min(), cons_mask.min()) * 100
    cons_max = max(cons_raw.max(), cons_mask.max()) * 100
    dz_min   = min(dz_raw.min(),   dz_mask.min())
    dz_max   = max(dz_raw.max(),   dz_mask.max())
    nv_min   = min(nv_raw.min(),   nv_mask.min())
    nv_max   = max(nv_raw.max(),   nv_mask.max())

    print(f'\ncons_rate  BEFORE mask : mean={cons_raw.mean()*100:6.2f}%  '
          f'median={np.median(cons_raw)*100:6.2f}%  std={cons_raw.std()*100:5.2f}%  '
          f'min={cons_raw.min()*100:6.2f}%  max={cons_raw.max()*100:6.2f}%')
    print(f'cons_rate  AFTER  mask : mean={cons_mask.mean()*100:6.2f}%  '
          f'median={np.median(cons_mask)*100:6.2f}%  std={cons_mask.std()*100:5.2f}%  '
          f'min={cons_mask.min()*100:6.2f}%  max={cons_mask.max()*100:6.2f}%')
    print(f'Δ mean cons_rate (mask − raw) = '
          f'{(cons_mask.mean() - cons_raw.mean())*100:+.2f} pp')

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    ROW_LABELS = ['BEFORE face mask (opacity only)', 'AFTER face mask']
    for row_idx, (cons, dz, nv, label, color) in enumerate([
        (cons_raw,  dz_raw,  nv_raw,  ROW_LABELS[0], 'steelblue'),
        (cons_mask, dz_mask, nv_mask, ROW_LABELS[1], 'darkorange'),
    ]):
        ax = axes[row_idx, 0]
        ax.hist(cons * 100, bins=30, range=(cons_min, cons_max),
                color=color, edgecolor='k')
        ax.axvline(cons.mean() * 100, color='r', ls='--',
                   label=f'mean={cons.mean()*100:.1f}%')
        ax.set_xlabel('Consistency rate (%)')
        ax.set_ylabel('# samples')
        ax.set_title(f'[{label}]  Per-sample cons_rate (n={len(rows)})')
        ax.set_xlim(cons_min, cons_max)
        ax.legend()

        ax = axes[row_idx, 1]
        ax.hist(dz, bins=30, range=(dz_min, dz_max),
                color=color, edgecolor='k', alpha=0.85)
        ax.axvline(dz.mean(), color='r', ls='--',
                   label=f'mean={dz.mean():.4f}')
        ax.set_xlabel('Mean |Δz| (world units)')
        ax.set_ylabel('# samples')
        ax.set_title(f'[{label}]  Per-sample mean|Δz|')
        ax.set_xlim(dz_min, dz_max)
        ax.legend()

        ax = axes[row_idx, 2]
        ax.scatter(nv, cons * 100, s=12, alpha=0.6, color=color)
        ax.set_xlabel('# visible pixels (after tight mask)')
        ax.set_ylabel('cons_rate (%)')
        ax.set_title(f'[{label}]  Visibility vs consistency')
        ax.set_xlim(nv_min, nv_max)
        ax.set_ylim(cons_min, cons_max)

    plt.tight_layout()
    plt.show()

    # Per-sample delta
    delta = (cons_mask - cons_raw) * 100
    fig2, ax2 = plt.subplots(1, 2, figsize=(12, 4))
    ax2[0].hist(delta, bins=40, color='purple', edgecolor='k')
    ax2[0].axvline(0, color='k', ls='-', lw=0.8)
    ax2[0].axvline(delta.mean(), color='r', ls='--',
                   label=f'mean Δ = {delta.mean():+.2f} pp')
    ax2[0].set_xlabel('Δ cons_rate  (mask − raw, percentage points)')
    ax2[0].set_ylabel('# samples')
    ax2[0].set_title('Per-sample Δ cons_rate')
    ax2[0].legend()

    ax2[1].scatter(cons_raw * 100, cons_mask * 100, s=10, alpha=0.5)
    lim = [cons_min, cons_max]
    ax2[1].plot(lim, lim, 'k--', lw=0.8)
    ax2[1].set_xlim(lim); ax2[1].set_ylim(lim)
    ax2[1].set_xlabel('cons_rate BEFORE mask (%)')
    ax2[1].set_ylabel('cons_rate AFTER mask (%)')
    ax2[1].set_title('Per-sample scatter (points above y=x → mask improved)')
    plt.tight_layout()
    plt.show()


## 2.6 Mask gain analysis — does the face mask help?


In [ ]:

# Single source of truth for the dual CSV
dual_csv = MULTIVIEW_DIR / "consistency_summary_both.csv"
assert dual_csv.exists(), (
    f"{dual_csv} not found — run section 2.5 first to generate it."
)

df = pd.read_csv(dual_csv)
print(f"[5.4c] Loaded {len(df)} rows from {dual_csv.name}")

# Gains
df["cons_gain"] = df["cons_rate_mask"] - df["cons_rate_raw"]   # >0 = mask helped
df["dz_gain"]   = df["mean_dz_raw"]    - df["mean_dz_mask"]    # >0 = mask helped

n_pos_cons = int((df["cons_gain"] > 0).sum())
n_pos_dz   = int((df["dz_gain"]   > 0).sum())

print(f"\ncons_gain  mean={df['cons_gain'].mean()*100:+.2f} pp   "
      f"median={df['cons_gain'].median()*100:+.2f} pp   "
      f"pos/total = {n_pos_cons}/{len(df)} ({100*n_pos_cons/len(df):.1f}%)")
print(f"dz_gain    mean={df['dz_gain'].mean():+.5f}   "
      f"median={df['dz_gain'].median():+.5f}   "
      f"pos/total = {n_pos_dz}/{len(df)} ({100*n_pos_dz/len(df):.1f}%)")

# Pass / fail summary line (the three criteria from the markdown above)
def _verdict(ok):
    return "PASS" if ok else "FAIL"
print(f"\n[pass check]")
print(f"  mean(cons_gain) > 0      : {_verdict(df['cons_gain'].mean() > 0)}")
print(f"  mean(dz_gain)   > 0      : {_verdict(df['dz_gain'].mean()   > 0)}")
print(f"  >50% samples above y=x   : {_verdict(n_pos_cons > len(df) / 2)}")

# ---- Plots (3 panels) ----
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# 1) consistency gain distribution (percentage points)
cg_pp = df["cons_gain"] * 100
ax[0].hist(cg_pp, bins=40, color="steelblue", edgecolor="k")
ax[0].axvline(0, color="k", lw=0.8)
ax[0].axvline(cg_pp.mean(), color="r", ls="--",
              label=f"mean = {cg_pp.mean():+.2f} pp")
ax[0].set_xlabel("cons_gain (mask − raw, percentage points)")
ax[0].set_ylabel("# samples")
ax[0].set_title(f"cons_gain   n_pos={n_pos_cons}/{len(df)}")
ax[0].legend()

# 2) depth error gain distribution (world units)
ax[1].hist(df["dz_gain"], bins=40, color="seagreen", edgecolor="k")
ax[1].axvline(0, color="k", lw=0.8)
ax[1].axvline(df["dz_gain"].mean(), color="r", ls="--",
              label=f"mean = {df['dz_gain'].mean():+.5f}")
ax[1].set_xlabel("dz_gain (raw − mask, world units)")
ax[1].set_ylabel("# samples")
ax[1].set_title(f"dz_gain   n_pos={n_pos_dz}/{len(df)}")
ax[1].legend()

# 3) scatter raw vs mask (percentages for readability)
raw_pct  = df["cons_rate_raw"]  * 100
mask_pct = df["cons_rate_mask"] * 100
ax[2].scatter(raw_pct, mask_pct, s=6, alpha=0.5, color="darkorange")
lo = min(raw_pct.min(), mask_pct.min())
hi = max(raw_pct.max(), mask_pct.max())
ax[2].plot([lo, hi], [lo, hi], "r--", lw=0.8, label="y = x")
ax[2].set_xlim(lo, hi); ax[2].set_ylim(lo, hi)
ax[2].set_xlabel("cons_rate RAW (%)")
ax[2].set_ylabel("cons_rate MASK (%)")
ax[2].set_title("raw vs mask   (above y=x → mask improved)")
ax[2].legend(loc="lower right")

plt.tight_layout()
plt.show()

# Print the 5 samples where the mask HURT the most (negative cons_gain)
worst_hurt = df.sort_values("cons_gain").head(5)
print("\n5 samples where mask reduced cons_rate the most "
      "(candidates for re-checking mask quality):")
for _, r in worst_hurt.iterrows():
    print(f"  {r['name']:40s}  cons_gain={r['cons_gain']*100:+6.2f} pp  "
          f"(raw={r['cons_rate_raw']*100:5.2f}% -> mask={r['cons_rate_mask']*100:5.2f}%)")


## 3.1 Build face masks from `cropped_faces`


In [ ]:

# PROJECT_ROOT / CROPPED_FACES_DIR / _build_face_mask_from_cropped come from
# section 0 (setup) + section 2.3 (FG mask). No local redefinition — single source of truth.
assert CROPPED_FACES_DIR.exists(), f"Missing {CROPPED_FACES_DIR}"
assert DATASET_DIR.exists(),       f"Missing {DATASET_DIR}"

# 6.1 uses a LOOSER profile than the §5 consistency mask:
#   - WHITE_TOL=12 keeps matting boundary + hair (dataset mask should include
#     the whole person region, not just the tight face core)
#   - erode_px=0 so the mask edge isn't pulled in
# We pass these as explicit kwargs to _build_face_mask_from_cropped — the
# function itself stays canonical.
FG_WHITE_TOL = 12
FG_K_CLOSE   = 7
FG_K_OPEN    = 3
FG_ERODE_PX  = 0
HR_SIZE      = 1024
LR_SIZE      = 256


def process_split(split: str):
    hr_dir      = DATASET_DIR / split / "depth"
    mask_dir    = DATASET_DIR / split / "mask"
    mask_lr_dir = DATASET_DIR / split / "mask_lr"
    mask_dir.mkdir(parents=True, exist_ok=True)
    mask_lr_dir.mkdir(parents=True, exist_ok=True)

    stems = sorted(p.stem for p in hr_dir.glob("*.png"))
    print(f"[6.1 {split}] {len(stems)} samples")

    missing, n_done, fg_ratios = [], 0, []
    for stem in stems:
        # Unified function — call twice (HR + LR). The LR call re-runs the morphology
        # on the source-resolution RGB, which is cheap relative to everything else.
        m_hr = _build_face_mask_from_cropped(
            stem, res=HR_SIZE,
            white_tol=FG_WHITE_TOL, k_close=FG_K_CLOSE,
            k_open=FG_K_OPEN, erode_px=FG_ERODE_PX,
        )
        if m_hr is None:
            missing.append(stem)
            continue
        m_lr = _build_face_mask_from_cropped(
            stem, res=LR_SIZE,
            white_tol=FG_WHITE_TOL, k_close=FG_K_CLOSE,
            k_open=FG_K_OPEN, erode_px=FG_ERODE_PX,
        )
        # Save as 0/255 uint8 PNG (bool -> uint8)
        Image.fromarray((m_hr.astype(np.uint8) * 255), mode="L").save(mask_dir    / f"{stem}.png")
        Image.fromarray((m_lr.astype(np.uint8) * 255), mode="L").save(mask_lr_dir / f"{stem}.png")
        fg_ratios.append(float(m_hr.mean()))
        n_done += 1

    if fg_ratios:
        a = np.array(fg_ratios)
        print(f"  wrote {n_done} masks  |  fg_ratio mean={a.mean():.3f} "
              f"min={a.min():.3f} max={a.max():.3f}")
    if missing:
        print(f"  WARNING: {len(missing)} samples with no cropped face — first 5: {missing[:5]}")


for split in ("train", "val"):
    process_split(split)

# Sanity-check viz: one train sample
stems = sorted(p.stem for p in (DATASET_DIR / "train" / "depth").glob("*.png"))
if stems:
    stem = stems[0]
    rgb_vis  = np.array(Image.open(CROPPED_FACES_DIR / f"{stem}.png").convert("RGB"))
    mask_vis = np.array(Image.open(DATASET_DIR / "train" / "mask" / f"{stem}.png"))
    hr_depth = np.array(Image.open(DATASET_DIR / "train" / "depth" / f"{stem}.png"))
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(rgb_vis);                ax[0].set_title(f"cropped {stem}"); ax[0].axis("off")
    ax[1].imshow(mask_vis, cmap="gray");  ax[1].set_title("fg mask 1024");    ax[1].axis("off")
    ax[2].imshow(hr_depth, cmap="magma"); ax[2].set_title("HR depth");        ax[2].axis("off")
    overlay = (hr_depth.astype(np.float32) / max(hr_depth.max(), 1)) * (mask_vis > 127)
    ax[3].imshow(overlay, cmap="magma");  ax[3].set_title("masked HR");       ax[3].axis("off")
    plt.tight_layout(); plt.show()

print("\n[6.1] Foreground masks ready at data/dataset/{train,val}/mask/ + mask_lr/")
print("      Using unified _build_face_mask_from_cropped with loose profile "
      f"(WHITE_TOL={FG_WHITE_TOL}, erode_px={FG_ERODE_PX})")


## 3.2 Mask debug — per-sample 8-panel diagnostic


### 3.2.1 Config & paths


In [ ]:

SPLITS        = ("train", "val")
OPACITY_TAU   = 0.5     # opacity is uint8 0..255, threshold as ratio
ERODE_OP_PX   = 0       # >0 to erode, requires scipy
FORCE         = False   # True = ignore cache, regenerate everything
MAX_PER_SPLIT = None    # Set a small number for debugging
DPI           = 80

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "FaceLift" else Path.cwd().resolve()
DEBUG_ROOT   = PROJECT_ROOT / "data" / "mask_debug"
DEBUG_ROOT.mkdir(parents=True, exist_ok=True)

# Optional deps — runs fine without them
try:
    from scipy.ndimage import binary_erosion
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

try:
    from skimage.segmentation import find_boundaries
    _HAS_SKI = True
except Exception:
    _HAS_SKI = False


### 3.2.2 Utilities — read sample / build mask / compute metrics


In [ ]:
def _boundary(mask_bool):
    if _HAS_SKI:
        return find_boundaries(mask_bool, mode="outer")
    m = mask_bool.astype(np.uint8)
    pad = np.pad(m, 1, mode="constant")
    nb = pad[:-2, 1:-1] + pad[2:, 1:-1] + pad[1:-1, :-2] + pad[1:-1, 2:]
    return (m == 0) & (nb > 0)


def _load_sample(split, stem):
    rgb_p  = DATASET_DIR / split / "image"   / f"{stem}.png"
    dep_p  = DATASET_DIR / split / "depth"   / f"{stem}.png"
    op_p   = DATASET_DIR / split / "opacity" / f"{stem}.png"
    face_p = DATASET_DIR / split / "mask"    / f"{stem}.png"
    missing = [p.name for p in (rgb_p, dep_p, op_p, face_p) if not p.exists()]
    if missing:
        return None, missing
    rgb   = np.array(Image.open(rgb_p).convert("RGB"))
    depth = np.array(Image.open(dep_p))                # uint16
    op    = np.array(Image.open(op_p).convert("L"))    # uint8
    face  = np.array(Image.open(face_p).convert("L"))  # uint8
    return (rgb, depth, op, face), []


def _compute_masks(opacity_u8, face_u8):
    mask_op = opacity_u8 > int(OPACITY_TAU * 255)
    if ERODE_OP_PX > 0 and _HAS_SCIPY:
        mask_op = binary_erosion(mask_op, iterations=ERODE_OP_PX)
    mask_face  = face_u8 > 127
    mask_final = mask_op & mask_face
    return mask_op, mask_face, mask_final


def _metrics(mask_op, mask_face, mask_final):
    op_r    = float(mask_op.mean())
    face_r  = float(mask_face.mean())
    final_r = float(mask_final.mean())
    inter   = float((mask_op & mask_face).sum())
    union   = float((mask_op | mask_face).sum())
    iou     = inter / max(union, 1.0)
    keep_r  = float(mask_final.sum()) / max(float(mask_face.sum()), 1.0)
    return op_r, face_r, final_r, iou, keep_r


### 3.2.3 8-panel figure renderer


In [ ]:
def _render_figure(rgb, depth, opacity, mask_op, mask_face, mask_final,
                   op_r, face_r, final_r, iou, keep_r, title, out_path):
    depth_vis = depth.astype(np.float32) / max(float(depth.max()), 1.0)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes[0, 0].imshow(rgb);                      axes[0, 0].set_title("RGB")
    axes[0, 1].imshow(opacity, cmap="gray");     axes[0, 1].set_title("opacity")
    axes[0, 2].imshow(depth_vis, cmap="magma");  axes[0, 2].set_title("depth")
    axes[0, 3].imshow(mask_op, cmap="gray");     axes[0, 3].set_title(f"mask_op  {op_r:.2f}")
    axes[1, 0].imshow(mask_face, cmap="gray");   axes[1, 0].set_title(f"mask_face  {face_r:.2f}")
    axes[1, 1].imshow(mask_final, cmap="gray");  axes[1, 1].set_title(f"mask_final  {final_r:.2f}")

    overlay = rgb.copy()
    overlay[_boundary(mask_final)] = [255, 0, 0]
    axes[1, 2].imshow(overlay);                  axes[1, 2].set_title("RGB + boundary")

    axes[1, 3].imshow(depth_vis * mask_final, cmap="magma")
    axes[1, 3].set_title(f"depth*mask  IoU={iou:.2f}  keep={keep_r:.2f}")

    for ax in axes.flat:
        ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


### 3.2.4 Main loop (with cache skip)


In [ ]:
def debug_split(split):
    out_dir = DEBUG_ROOT / split
    out_dir.mkdir(parents=True, exist_ok=True)

    depth_dir = DATASET_DIR / split / "depth"
    stems = sorted(p.stem for p in depth_dir.glob("*.png"))
    if MAX_PER_SPLIT is not None:
        stems = stems[:MAX_PER_SPLIT]

    rows = []
    n_render = n_cached = n_missing = 0
    missing_info = []

    for stem in stems:
        out_png = out_dir / f"{stem}.png"
        loaded, missing = _load_sample(split, stem)
        if loaded is None:
            n_missing += 1
            missing_info.append((stem, missing))
            continue
        rgb, depth, op, face = loaded
        mask_op, mask_face, mask_final = _compute_masks(op, face)
        op_r, face_r, final_r, iou, keep_r = _metrics(mask_op, mask_face, mask_final)

        rows.append({
            "split":   split,
            "stem":    stem,
            "op_r":    round(op_r,    4),
            "face_r":  round(face_r,  4),
            "final_r": round(final_r, 4),
            "iou":     round(iou,     4),
            "keep_r":  round(keep_r,  4),
        })

        # Cache hit: skip redraw
        if out_png.exists() and not FORCE:
            n_cached += 1
            continue

        title = f"[{split}] {stem}   op={op_r:.2f} face={face_r:.2f} final={final_r:.2f} IoU={iou:.2f} keep={keep_r:.2f}"
        _render_figure(rgb, depth, op, mask_op, mask_face, mask_final,
                       op_r, face_r, final_r, iou, keep_r, title, out_png)
        n_render += 1

    print(f"[{split}] total={len(stems)}  rendered={n_render}  cached={n_cached}  missing={n_missing}")
    if missing_info:
        print(f"  first missing: {missing_info[:3]}")
    return rows


all_rows = []
for sp in SPLITS:
    all_rows.extend(debug_split(sp))


### 3.2.5 CSV export + worst-10 printout


In [ ]:
if all_rows:
    all_rows.sort(key=lambda r: r["keep_r"])
    csv_path = DEBUG_ROOT / "mask_debug_summary.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(all_rows[0].keys()))
        w.writeheader()
        w.writerows(all_rows)
    print(f"summary CSV: {csv_path}  ({len(all_rows)} rows)")

    keep = np.array([r["keep_r"] for r in all_rows])
    iou  = np.array([r["iou"]    for r in all_rows])
    print(f"keep_r  mean={keep.mean():.3f}  min={keep.min():.3f}  p05={np.percentile(keep,5):.3f}  p50={np.percentile(keep,50):.3f}")
    print(f"iou     mean={iou.mean():.3f}  min={iou.min():.3f}  p05={np.percentile(iou,5):.3f}  p50={np.percentile(iou,50):.3f}")

    print("\n=== 10 worst by keep_r ===")
    for r in all_rows[:10]:
        print(f"  keep={r['keep_r']:.2f}  iou={r['iou']:.2f}  {r['split']}/{r['stem']}")

print(f"\nfigures -> {DEBUG_ROOT}")
print("Re-render one: delete data/mask_debug/<split>/<stem>.png and re-run; regenerate all: FORCE=True")


## 4.1 Thumbnail browser → `data/to_delete.txt`


In [ ]:
# 4.1  Paginated cropped_faces thumbnail browser
# Three modes, pick with the flags below:
#   ALL_PAGES=False + SAVE_TO_DISK=False -> show ONE page (set PAGE to browse)
#   ALL_PAGES=True                        -> show ALL pages stacked in output (heavy but no file I/O)
#   SAVE_TO_DISK=True                     -> dump all pages as PNGs to data/to_delete_browse/
# Then copy glasses-wearing stems into data/to_delete.txt, one per line.

DATA_ROOT   = (PROJECT_ROOT / "data").resolve()
CROPPED     = DATA_ROOT / "cropped_faces"
TO_DEL_TXT  = DATA_ROOT / "to_delete.txt"
BROWSE_DIR  = DATA_ROOT / "to_delete_browse"

PAGE          = 0       # only used when ALL_PAGES=False and SAVE_TO_DISK=False
PAGE_SIZE     = 64      # 8 x 8
COLS          = 8
ALL_PAGES     = False   # True = render every page inline (slow + tall notebook output)
SAVE_TO_DISK  = True    # True = save each page as PNG to BROWSE_DIR, don't render inline

assert CROPPED.exists(), f"Missing {CROPPED}"

# Stub to_delete.txt if missing
_TEMPLATE = """# data/to_delete.txt
# One sample stem per line (no .png extension).
# Lines starting with # are ignored, blank lines too.
#
# Examples (uncomment and replace with real stems from cell 4.1):
#   ffhq_12163_face
#   1 (1)_face
#
# After editing, re-run cell 4.2.
"""
if not TO_DEL_TXT.exists():
    TO_DEL_TXT.write_text(_TEMPLATE, encoding="utf-8")
    print(f"[4.1] created stub: {TO_DEL_TXT}")

samples = sorted(CROPPED.glob("*.png"))
n_total = len(samples)
n_pages = max(1, (n_total + PAGE_SIZE - 1) // PAGE_SIZE)
n_already = sum(1 for l in TO_DEL_TXT.read_text(encoding="utf-8").splitlines()
                if l.strip() and not l.strip().startswith("#"))
print(f"total samples={n_total}   pages={n_pages}   PAGE_SIZE={PAGE_SIZE}")
print(f"to_delete.txt currently has {n_already} entries -> {TO_DEL_TXT}")

def _render_page(page_idx, save_path=None):
    start = page_idx * PAGE_SIZE
    end   = min(start + PAGE_SIZE, n_total)
    batch = samples[start:end]
    if not batch:
        return
    rows = (len(batch) + COLS - 1) // COLS
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 1.6, rows * 1.9))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    fig.suptitle(f"Page {page_idx+1}/{n_pages}  (samples {start}..{end-1})", fontsize=10)
    for i, p in enumerate(batch):
        ax = axes[i]
        try:
            ax.imshow(Image.open(p))
        except Exception as e:
            ax.text(0.5, 0.5, f"ERR\n{e}", ha="center", va="center")
        ax.set_title(p.stem, fontsize=6)
        ax.axis("off")
    for j in range(len(batch), len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=100, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

if SAVE_TO_DISK:
    BROWSE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"\n[4.1] Saving {n_pages} pages as PNG to: {BROWSE_DIR}")
    for p in range(n_pages):
        out = BROWSE_DIR / f"page_{p+1:02d}_of_{n_pages:02d}.png"
        _render_page(p, save_path=out)
        print(f"  wrote {out.name}")
    print("[4.1] Open that folder in your file browser, flip through the PNGs,")
    print("      and copy stems of unwanted samples into to_delete.txt.")
elif ALL_PAGES:
    print(f"\n[4.1] Rendering all {n_pages} pages inline (this is heavy)...")
    for p in range(n_pages):
        _render_page(p)
else:
    print(f"\n[4.1] Showing page {PAGE+1}/{n_pages}  (set PAGE = {PAGE+1} and re-run to advance,")
    print(f"      or set ALL_PAGES=True, or SAVE_TO_DISK=True)")
    _render_page(PAGE)


## 4.2 Safe delete — dry-run by default

Set `CONFIRM_DELETE = True` after reviewing the dry-run output.


In [ ]:
# 4.2  Safe delete: read data/to_delete.txt, plan + dry-run, then execute on confirm.
import csv, shutil

CONFIRM_DELETE = False   # <-- flip to True to ACTUALLY delete; keep False for dry-run

DATA_ROOT  = (PROJECT_ROOT / "data").resolve()
TO_DEL_TXT = DATA_ROOT / "to_delete.txt"

# If to_delete.txt is missing, create a stub with instructions and exit politely.
_TEMPLATE = """# data/to_delete.txt
# One sample stem per line (no .png extension).
# Lines starting with # are ignored, blank lines too.
#
# Examples (uncomment and replace with real stems from cell 4.1):
#   ffhq_12163_face
#   1 (1)_face
#
# After editing, re-run cell 4.2.
"""
if not TO_DEL_TXT.exists():
    TO_DEL_TXT.write_text(_TEMPLATE, encoding="utf-8")
    print(f"[4.2] created empty template: {TO_DEL_TXT}")
    print("[4.2] Edit it (one stem per line, no .png), then re-run this cell.")
    raise SystemExit  # stop here, don't error

# --- Read stems ---
raw_lines = TO_DEL_TXT.read_text(encoding="utf-8").splitlines()
stems = []
for l in raw_lines:
    s = l.strip()
    if not s or s.startswith("#"):
        continue
    if s.lower().endswith(".png"):
        s = s[:-4]
    stems.append(s)
stems = sorted(set(stems))
stem_set = set(stems)
print(f"[cleanup] to-delete list: {len(stems)} unique stems")

if not stems:
    print(f"[4.2] {TO_DEL_TXT} has no entries (only comments / blank lines).")
    print("[4.2] Add stems via cell 4.1, then re-run.")
    raise SystemExit

# --- Per-sample target collector ---
def _paths_for(stem):
    """All on-disk locations a single sample touches."""
    raw_stem = stem[:-5] if stem.endswith("_face") else stem
    out = []

    # single-file pngs in flat per-sample dirs
    for sub in ["cropped_faces", "rgb_rendered",
                "depth_maps", "normal_maps", "opacity_maps"]:
        p = DATA_ROOT / sub / f"{stem}.png"
        if p.exists():
            out.append(p)

    # raw_faces uses the un-suffixed stem; try common extensions
    for ext in (".jpg", ".jpeg", ".png"):
        p = DATA_ROOT / "raw_faces" / f"{raw_stem}{ext}"
        if p.exists():
            out.append(p)

    # postprocessed/{modality}/<stem>.png
    for m in ["rgb", "depth", "normal", "opacity"]:
        p = DATA_ROOT / "postprocessed" / m / f"{stem}.png"
        if p.exists():
            out.append(p)

    # dataset/{split}/{modality}/<stem>.png
    for split in ["train", "val"]:
        for m in ["image", "depth", "normal", "opacity",
                  "depth_lr_8bit", "depth_lr_16bit"]:
            p = DATA_ROOT / "dataset" / split / m / f"{stem}.png"
            if p.exists():
                out.append(p)

    # mask_debug/{split}/<stem>*  (PNG / NPZ / whatever)
    for split in ["train", "val"]:
        d = DATA_ROOT / "mask_debug" / split
        if d.exists():
            for p in d.glob(f"{stem}*"):
                out.append(p)

    # per-sample directories
    for sub in ["splats", "multiview_preview"]:
        d = DATA_ROOT / sub / stem
        if d.exists():
            out.append(d)

    return out

# --- Build plan ---
plan = {}
total_size = 0
n_missing = 0
for stem in stems:
    ts = _paths_for(stem)
    if not ts:
        n_missing += 1
        print(f"  [!] no files found for stem: {stem!r}")
        continue
    plan[stem] = ts
    for p in ts:
        try:
            if p.is_dir():
                for f in p.rglob("*"):
                    if f.is_file():
                        total_size += f.stat().st_size
            else:
                total_size += p.stat().st_size
        except Exception:
            pass

# --- CSV row removal plan ---
csv_targets = [
    DATA_ROOT / "multiview_preview" / "consistency_summary.csv",
    DATA_ROOT / "multiview_preview" / "consistency_summary_both.csv",
]
csv_plan = {}   # csv_path -> (all_rows, hits, fieldnames)
for csv_path in csv_targets:
    if not csv_path.exists():
        continue
    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        rows = list(rdr)
        fieldnames = rdr.fieldnames or []
    hits = [r for r in rows if r.get("name") in stem_set]
    csv_plan[csv_path] = (rows, hits, fieldnames)

# --- Print plan ---
print(f"\n[cleanup] PLAN")
print(f"  samples found     : {len(plan)}")
print(f"  samples missing   : {n_missing}")
print(f"  total paths       : {sum(len(v) for v in plan.values())}")
print(f"  total size        : {total_size/(1024**2):.1f} MB")
print(f"  CSV rows to drop  :")
for csv_path, (rows, hits, _) in csv_plan.items():
    print(f"    {csv_path.name:35s}  {len(hits)} / {len(rows)}")

if plan:
    example = next(iter(plan))
    print(f"\n  Example for {example!r}:")
    for p in plan[example][:12]:
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"    [{kind}] {p.relative_to(DATA_ROOT)}")
    if len(plan[example]) > 12:
        print(f"    ... ({len(plan[example]) - 12} more)")

# --- Execute or dry-run ---
if not CONFIRM_DELETE:
    print("\n[cleanup] DRY-RUN ONLY. Set CONFIRM_DELETE = True and re-run to actually delete.")
else:
    print("\n[cleanup] EXECUTING DELETE ...")
    n_del_files, n_del_dirs = 0, 0
    for stem, ts in plan.items():
        for p in ts:
            try:
                if p.is_dir():
                    shutil.rmtree(p)
                    n_del_dirs += 1
                else:
                    p.unlink()
                    n_del_files += 1
            except Exception as e:
                print(f"  [ERR] {p}: {e}")

    for csv_path, (rows, hits, fieldnames) in csv_plan.items():
        if not hits:
            continue
        keep = [r for r in rows if r.get("name") not in stem_set]
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(keep)
        print(f"  updated {csv_path.name}: dropped {len(hits)} rows, kept {len(keep)}")

    print(f"\n[cleanup] DONE. Removed {n_del_files} files, {n_del_dirs} directories")
    print(f"[cleanup] Next: run cell 6 to rebuild dataset/train + dataset/val + manifest.json")


## 5.0 Path bootstrap for low-consistency cleanup


In [ ]:

# Notebook lives at FaceLift/facelift_pipeline.ipynb,
# so its cwd is D:\zmm\facelift_pipeline\FaceLift → project root is one level up.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "FaceLift" else Path.cwd().resolve()
MV_DIR       = PROJECT_ROOT / "data" / "multiview_preview"
SPLAT_DIR    = PROJECT_ROOT / "data" / "splats"     # actual splat location (FaceLift/splats)
CSV_PATH     = MV_DIR / "consistency_summary.csv"

assert SPLAT_DIR.exists(), f"SPLAT_DIR not found: {SPLAT_DIR}"
assert MV_DIR.exists(),    f"MV_DIR not found: {MV_DIR} (run Step 5.4 first)"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SPLAT_DIR    = {SPLAT_DIR}")
print(f"MV_DIR       = {MV_DIR}")
print(f"CSV_PATH     = {CSV_PATH}  (exists={CSV_PATH.exists()})")

## 5.1 Scan low-consistency samples (visualize only)


In [ ]:
# Self-contained path setup — derives everything from `config` (loaded in cell 0),
# so this cell can run even if section 2 has not been executed in this kernel.
MV_DIR    = (Path(config['paths']['depth_output']).parent / 'multiview_preview').resolve()
CSV_PATH  = MV_DIR / "consistency_summary.csv"

# ---- Thresholds ----
CONS_RATE_THR = 0.78          # cons_rate below this = bad
MEAN_DZ_MODE  = "percentile"  # 'percentile' or 'absolute'
MEAN_DZ_PCT   = 95
MEAN_DZ_ABS   = 0.05
MAX_DELETE    = None          # None = delete all bad; int = only worst N
SHOW_PREVIEW  = 0             # Visualize first N bad samples; 0 = list only, no render

assert CSV_PATH.exists(), f"Cannot find {CSV_PATH}, run section 2.4 (Batch consistency) first to generate the summary"
df = pd.read_csv(CSV_PATH)
print(f"Total samples: {len(df)}  columns: {list(df.columns)}")

# 2.4 already aggregated by name; take the worse row as fallback
agg = df.groupby("name", as_index=False).agg(
    cons_rate=("cons_rate", "min"),
    mean_dz  =("mean_dz",   "max"),
)

if MEAN_DZ_MODE == "percentile":
    dz_cut = float(np.percentile(agg["mean_dz"].values, MEAN_DZ_PCT))
else:
    dz_cut = MEAN_DZ_ABS
print(f"cons_rate threshold: < {CONS_RATE_THR}   mean_dz threshold: > {dz_cut:.4f}")

bad_mask = (agg["cons_rate"] < CONS_RATE_THR) | (agg["mean_dz"] > dz_cut)
bad = (agg[bad_mask]
       .sort_values(["cons_rate", "mean_dz"], ascending=[True, False])
       .reset_index(drop=True))
print(f"Flagged as bad: {len(bad)} / {len(agg)}  "
      f"({len(bad)/max(1,len(agg))*100:.1f}%)")

# Apply MAX_DELETE cap (only the worst N)
if MAX_DELETE is not None and len(bad) > MAX_DELETE:
    bad = bad.head(MAX_DELETE).reset_index(drop=True)
    print(f"MAX_DELETE={MAX_DELETE} active, keeping only worst {len(bad)}  as deletion candidates")

try:
    display(bad.head(20))
except NameError:
    print(bad.head(20))

# ---- Visualization (optional, only if SHOW_PREVIEW > 0) ----
# On-demand re-render at yaw=0 (no disk read). Needs get_camera_at_yaw from §2.1 and
# render_opencv_cam + render_with_custom_colors from §1.1.
def _render_one_preview(name):
    ply = SPLAT_DIR / name / "gaussians.ply"
    if not ply.exists():
        return None, None, None
    pc = GS_GaussianModel(sh_degree=3)
    pc.load_ply(str(ply))
    pc = pc.to(RENDER_DEVICE)
    n_pts = pc.get_xyz.shape[0]
    with torch.no_grad():
        fxfycxcy, c2w = get_camera_at_yaw(yaw_deg=0)
        # RGB
        rgb = render_opencv_cam(pc, RENDER_RES, RENDER_RES, c2w, fxfycxcy,
                                bg_color=(1., 1., 1.))['render']
        rgb = (rgb.detach().cpu().numpy() * 255).clip(0, 255) \
                 .astype(np.uint8).transpose(1, 2, 0)
        # Opacity
        white = torch.ones(n_pts, 3, dtype=torch.float32, device=RENDER_DEVICE)
        op_r  = render_with_custom_colors(pc, RENDER_RES, RENDER_RES, c2w, fxfycxcy,
                                          white, bg_color=(0., 0., 0.))
        op    = op_r[0].detach().cpu().numpy().clip(0, 1)
        # Depth (per-view norm, visualization only)
        vc  = Camera(C2W=c2w, fxfycxcy=fxfycxcy, h=RENDER_RES, w=RENDER_RES)
        w2c = vc.world_view_transform.T
        xyz_h = torch.cat([pc.get_xyz,
                           torch.ones(n_pts, 1, device=RENDER_DEVICE)], dim=1)
        zs = (w2c @ xyz_h.T).T[:, 2]
        v  = zs > 0.1
        if v.any():
            d_min = float(zs[v].min()); d_max = float(zs[v].max())
        else:
            d_min, d_max = 0.0, 1.0
        d_rng = max(d_max - d_min, 1e-8)
        dn    = (1.0 - (zs - d_min) / d_rng).clamp(0, 1)
        d_col = dn.unsqueeze(1).expand(-1, 3)
        d_r   = render_with_custom_colors(pc, RENDER_RES, RENDER_RES, c2w, fxfycxcy,
                                          d_col, bg_color=(0., 0., 0.))
        depth = d_r[0].detach().cpu().numpy().clip(0, 1)
    del pc; torch.cuda.empty_cache()
    return rgb, depth, op

if SHOW_PREVIEW > 0 and len(bad) > 0:
    n = min(SHOW_PREVIEW, len(bad))
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        row = bad.iloc[i]
        rgb, depth, op = _render_one_preview(row["name"])
        if rgb is None:
            axes[i, 0].text(0.5, 0.5, "missing splat", ha="center", va="center")
            for j in range(3):
                axes[i, j].set_axis_off()
            continue
        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title(f"{row['name']}\ncons={row['cons_rate']:.3f}  dz={row['mean_dz']:.4f}",
                             fontsize=8)
        axes[i, 1].imshow(depth, cmap="turbo", vmin=0, vmax=1); axes[i, 1].set_title("depth yaw=0")
        axes[i, 2].imshow(op,    cmap="gray",  vmin=0, vmax=1); axes[i, 2].set_title("opacity yaw=0")
        for j in range(3):
            axes[i, j].set_axis_off()
    plt.tight_layout(); plt.show()

# ---- TO_DELETE: list of sample stems that 5.3 will remove ----
TO_DELETE = bad["name"].tolist()
print(f"\nTO_DELETE ready, contains {len(TO_DELETE)}  samples. Next: run §5.3.")
if TO_DELETE:
    print(f"  First 5: {TO_DELETE[:5]}")


## 5.2 Preview bad samples — self-contained re-render


In [ ]:
import torch
import cv2

# ---- (1) FaceLift repo + render deps (same as section 2.4) ----
if str(FACELIFT_REPO_PATH) not in sys.path:
    sys.path.insert(0, str(FACELIFT_REPO_PATH))

from gslrm.model.gaussians_renderer import (
    GaussianModel as GS_GaussianModel,
    GaussianRasterizationSettings,
    GaussianRasterizer,
    Camera,
    render_opencv_cam,
)

RENDER_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ---- (2) LOCAL helper (NOT shadowing the canonical one in section 1.1).
# This one returns (image, radii) because the preview callsite at line 119
# wants both; section 1.1 returns just the image.
def _rwc_local(pc, height, width, c2w, fxfycxcy, colors_precomp,
                              bg_color=(0.0, 0.0, 0.0)):
    viewpoint_camera = Camera(C2W=c2w, fxfycxcy=fxfycxcy, h=height, w=width)
    bg = torch.tensor(list(bg_color), dtype=torch.float32, device=c2w.device)
    raster_settings = GaussianRasterizationSettings(
        image_height=int(viewpoint_camera.h),
        image_width=int(viewpoint_camera.w),
        tanfovx=viewpoint_camera.tanfovX,
        tanfovy=viewpoint_camera.tanfovY,
        bg=bg,
        scale_modifier=1.0,
        viewmatrix=viewpoint_camera.world_view_transform,
        projmatrix=viewpoint_camera.full_proj_transform,
        sh_degree=0,
        campos=viewpoint_camera.camera_center,
        prefiltered=False,
        debug=False,
    )
    rasterizer = GaussianRasterizer(raster_settings=raster_settings)
    means3D    = pc.get_xyz
    means2D    = torch.zeros_like(means3D, dtype=means3D.dtype, device=means3D.device)
    rendered_image, radii = rasterizer(
        means3D=means3D, means2D=means2D, shs=None,
        colors_precomp=colors_precomp, opacities=pc.get_opacity,
        scales=pc.get_scaling, rotations=pc.get_rotation, cov3D_precomp=None,
    )
    return rendered_image, radii

def get_camera_at_yaw_local(yaw_deg=0.0):
    # Thin wrapper around the section 2.1 version, just in case that cell hasn't been re-run.
    return get_camera_at_yaw(yaw_deg=yaw_deg)

# ---- (3) Paths — derive what we can from config, require upstream for the rest ----
RES           = config['depth_export']['render_resolution']
CSV_PATH      = MULTIVIEW_DIR / "consistency_summary.csv"

# This cell needs functions / globals defined in earlier sections.
# If you are running this in a fresh kernel, run those cells first:
_missing = []
for _name in ['get_camera_at_yaw', 'CROPPED_FACES_DIR', '_build_face_mask_from_cropped']:
    if _name not in globals():
        _missing.append(_name)
if _missing:
    raise NameError(
        f"5.2 needs upstream globals not found in this kernel: {_missing}.\n"
        f"  - get_camera_at_yaw is defined by section 2.1\n"
        f"  - CROPPED_FACES_DIR + _build_face_mask_from_cropped are defined by section 2.3\n"
        f"Run those cells first, then re-run 5.2."
    )

CONS_RATE_THR = 0.81
N_SHOW        = 5   # Start with worst 5 to check mask quality; set None later for the full bad list

# ---- (4) Clean + normalize helper ----
# Clean NaN/inf + percentile normalization. When mask is given, percentiles are computed inside mask only,
# so hallucinated regions don't blow out the colormap.
def _clean_and_normalize(depth, mask=None, pct=(1.0, 99.0)):
    d = np.nan_to_num(depth.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    valid = (d > 0)
    if mask is not None:
        valid = valid & mask
    if not np.any(valid):
        return np.zeros_like(d), (0.0, 1.0)
    lo, hi = np.percentile(d[valid], pct)
    out = np.clip((d - lo) / (hi - lo + 1e-6), 0.0, 1.0)
    return out, (float(lo), float(hi))

# ---- (5) Select worst samples ----
df = pd.read_csv(CSV_PATH)
agg = df.groupby("name", as_index=False).agg(
    cons_rate=("cons_rate","min"), mean_dz=("mean_dz","max"))
bad = (agg[agg["cons_rate"] < CONS_RATE_THR]
       .sort_values("cons_rate").head(N_SHOW).reset_index(drop=True))
print(f"Previewing {len(bad)}  bad samples (cons_rate < {CONS_RATE_THR}）")

# ---- (6) Render yaw=0 + pull the unified face mask ----
def _render_yaw0(name):
    ply = SPLAT_DIR / name / "gaussians.ply"
    if not ply.exists():
        return None, None, None, None
    pc = GS_GaussianModel(sh_degree=3); pc.load_ply(str(ply)); pc = pc.to(RENDER_DEVICE)
    n = pc.get_xyz.shape[0]
    with torch.no_grad():
        fx, c2w = get_camera_at_yaw(yaw_deg=0)
        rgb = render_opencv_cam(pc, RES, RES, c2w, fx, bg_color=(1.,1.,1.))['render']
        rgb = (rgb.detach().cpu().numpy()*255).clip(0,255).astype(np.uint8).transpose(1,2,0)
        vc = Camera(C2W=c2w, fxfycxcy=fx, h=RES, w=RES)
        w2c = vc.world_view_transform.T
        xyz_h = torch.cat([pc.get_xyz, torch.ones(n,1,device=RENDER_DEVICE)], dim=1)
        zs = (w2c @ xyz_h.T).T[:,2]; v = zs > 0.1
        d_min = float(zs[v].min()) if v.any() else 0.0
        d_max = float(zs[v].max()) if v.any() else 1.0
        dn = (1.0 - (zs - d_min)/max(d_max-d_min,1e-8)).clamp(0,1)
        d_col = dn.unsqueeze(1).expand(-1,3)
        d_img = _rwc_local(pc, RES, RES, c2w, fx, d_col,
                           bg_color=(0.,0.,0.))[0]   # (3, H, W)
        depth = d_img[0].detach().cpu().numpy().clip(0, 1)            # (H, W)
    del pc; torch.cuda.empty_cache()

    # Unified face mask (same call as section 2.4's batch) + source RGB for the preview
    cropped_rgb, face_mask = _build_face_mask_from_cropped(name, res=RES, return_rgb=True)
    return rgb, depth, cropped_rgb, face_mask

# ---- 4-column preview:
#   Col 0: cropped_faces source (pre-FaceLift reference)
#   Col 1: rendered yaw=0 RGB (with hallucination)
#   Col 2: depth RAW (cleaned+normalized, no mask)
#   Col 3: depth MASKED (same d_norm, with face_mask applied)
# Col 2 / Col 3 share the same d_norm, normalization range set by in-face pixels,
# so hallucination outliers don't affect the colormap. Before/after is visually comparable.
if len(bad) == 0:
    print("No bad samples ✅")
else:
    fig, axes = plt.subplots(len(bad), 4, figsize=(14, 3.4*len(bad)))
    if len(bad) == 1: axes = axes[None, :]
    for i, row in bad.iterrows():
        rgb, depth, cropped_rgb, face_mask = _render_yaw0(row["name"])
        if rgb is None:
            for ax in axes[i]:
                ax.text(0.5,0.5,"missing ply",ha="center",va="center"); ax.set_axis_off()
            continue

        # Col 0: cropped_faces pre-FaceLift source (ground-truth reference)
        if cropped_rgb is not None:
            axes[i,0].imshow(cropped_rgb)
            axes[i,0].set_title(f"{row['name']}\ncropped_faces (source, pre-FaceLift)",
                                fontsize=8)
        else:
            axes[i,0].text(0.5, 0.5, "no cropped_faces\nsource on disk",
                           ha="center", va="center")
            axes[i,0].set_title(f"{row['name']}", fontsize=8)
        axes[i,0].set_axis_off()

        # Col 1: rendered RGB (with FaceLift hallucinated neck/shoulders)
        axes[i,1].imshow(rgb)
        axes[i,1].set_title(
            f"rendered yaw=0 (with hallucination)\n"
            f"cons={row.cons_rate:.2f}  dz={row.mean_dz:.3f}",
            fontsize=8)
        axes[i,1].set_axis_off()

        # Shared clean+normalize (percentile only uses face_mask pixels)
        d_norm, (lo, hi) = _clean_and_normalize(depth, mask=face_mask, pct=(1.0, 99.0))

        # Col 2: depth RAW — clean+normalize, not masked for display
        axes[i,2].imshow(d_norm, cmap="turbo", vmin=0, vmax=1)
        axes[i,2].set_title(f"depth RAW (clean+norm)\np1..p99 = [{lo:.3f}, {hi:.3f}]",
                            fontsize=8)
        axes[i,2].set_axis_off()

        # Col 3: depth MASKED — same d_norm, face_mask applied (NaN outside)
        if face_mask is not None:
            d_masked = np.where(face_mask, d_norm, np.nan)
            title3   = "depth MASKED (face mask)"
        else:
            d_masked = d_norm
            title3   = "depth MASKED (no mask on disk)"
        axes[i,3].imshow(d_masked, cmap="turbo", vmin=0, vmax=1)
        axes[i,3].set_title(title3, fontsize=8)
        axes[i,3].set_axis_off()

    plt.tight_layout(); plt.show()

# ---- Checklist (review worst 5 before tuning params) ----
# 1. Col 3 vs Col 2: are hair / shoulder ring / neck hallucinations actually masked out?
# 2. Col 3 itself: is the face core (nose / eye sockets / chin) intact, not over-eroded?
# 3. Col 3 vs Col 1: does the masked depth boundary follow the real face contour (check against hairline/jawline in rendered RGB)?
#
# Tuning guide — params are in §2.3 FACE_MASK_* globals, only one place to change:
#   • Hair/shoulders still in mask → FACE_MASK_ERODE_PX += 2 or FACE_MASK_WHITE_TOL += 5
#   • Face core clipped (nose/eyes cut) → FACE_MASK_ERODE_PX -= 2 or FACE_MASK_WHITE_TOL -= 5
#   • Small edge noise / fragments outside mask → FACE_MASK_K_OPEN += 2


## 5.3 Delete low-consistency samples (manual confirm)


In [ ]:

CONFIRM_DELETE = False   # Set to True after reviewing the visualization above

# Directories to clean (adjust to your project layout)
# Note: mask / mask_lr included too — generated from cropped_faces in §6.1, bad samples
# need their masks deleted as well, otherwise training fails on mismatched stems
TARGETS = [
    MV_DIR,                                                      # data/multiview_preview/<name>/
    PROJECT_ROOT / "data" / "postprocessed" / "rgb",
    PROJECT_ROOT / "data" / "postprocessed" / "depth",
    PROJECT_ROOT / "data" / "postprocessed" / "normal",
    PROJECT_ROOT / "data" / "postprocessed" / "opacity",
    PROJECT_ROOT / "data" / "rgb_rendered",
    PROJECT_ROOT / "data" / "depth_maps",
    PROJECT_ROOT / "data" / "normal_maps",
    PROJECT_ROOT / "data" / "opacity_maps",
    PROJECT_ROOT / "data" / "cropped_faces",
    SPLAT_DIR,
    # §6.1 face mask outputs — both splits, both full-res and LR copies
    PROJECT_ROOT / "data" / "dataset" / "train" / "mask",
    PROJECT_ROOT / "data" / "dataset" / "train" / "mask_lr",
    PROJECT_ROOT / "data" / "dataset" / "val"   / "mask",
    PROJECT_ROOT / "data" / "dataset" / "val"   / "mask_lr",
]

if not CONFIRM_DELETE:
    print("DRY RUN — printing only, nothing deleted. Set CONFIRM_DELETE=True to execute.\n")

deleted, missing = 0, 0
log = []
for name in TO_DELETE:
    for tgt in TARGETS:
        # Try subdirectory deletion first
        d = tgt / name
        # Fall back to filename deletion (with various extensions)
        candidates = [d] + list(tgt.glob(f"{name}.*"))
        for c in candidates:
            if c.exists():
                log.append(str(c))
                if CONFIRM_DELETE:
                    if c.is_dir(): shutil.rmtree(c)
                    else: c.unlink()
                deleted += 1
            else:
                missing += 1

print(f"{'Deleted' if CONFIRM_DELETE else 'Would delete'}: {deleted}  paths")
print(f"Unmatched paths: {missing}（normal — not all directories contain every sample）")
print("\nFirst 30 paths preview：")
for p in log[:30]: print(" ", p)

# Also remove bad rows from CSV
if CONFIRM_DELETE:
    df_clean = df[~df["name"].isin(TO_DELETE)].reset_index(drop=True)
    df_clean.to_csv(CSV_PATH, index=False)
    print(f"\nconsistency_summary.csv updated: {len(df)} → {len(df_clean)}  rows")

## 6. Rebuild dataset train/val split

Run after all deletions to refresh `dataset/train`, `dataset/val`, and `manifest.json`.


In [ ]:
# 6    Re-run reorganize_dataset.py to refresh dataset/train, dataset/val, manifest.json

script = (PROJECT_ROOT / "scripts" / "reorganize_dataset.py").resolve()
assert script.exists(), f"Missing {script}"

cmd = [sys.executable, str(script), "--val_ratio", "0.1", "--seed", "42"]
print(f"[rebuild] running: {' '.join(cmd)}")
print(f"[rebuild] cwd    : {PROJECT_ROOT}")

p = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
print(p.stdout)
if p.stderr:
    print("STDERR:", p.stderr)
if p.returncode != 0:
    raise SystemExit(f"reorganize_dataset.py failed with code {p.returncode}")

# Quick post-rebuild summary
manifest = PROJECT_ROOT / "data" / "dataset" / "manifest.json"
if manifest.exists():
    m = json.loads(manifest.read_text(encoding="utf-8"))
    # reorganize_dataset.py writes keys "train_samples" / "val_samples" (lists)
    # and "train_count" / "val_count" / "total" (ints). Prefer the counts.
    n_train = int(m.get("train_count", len(m.get("train_samples", []))))
    n_val   = int(m.get("val_count",   len(m.get("val_samples",   []))))
    n_total = int(m.get("total", n_train + n_val))
    print(f"\n[rebuild] manifest: train={n_train}  val={n_val}  total={n_total}")
